# Restriction-map geometry — MLP sheaf (fMRI + EEG)

What do the learned **data-dependent** restriction maps `\rho = MLP([h_src \| h_dst])`
actually *do*? This notebook rebuilds MLP-map checkpoints (`sheaf_mlp_maps=True`) for
fMRI and EEG, harvests the instantiated maps `(B, E, H, d_e)` from a probe batch, and
characterizes them:

1. **Norms** — Frobenius distribution; between-edge vs within-edge-across-windows variance.
2. **Effective rank** — participation ratio + singular spectra.
3. **Polar (rotation vs scaling)** — SVD-based `\rho = Q P` split (handles rectangular maps).
4. **Discriminability** — since the LSTM embeddings are near-collapsed, do the maps stay
   distinct or collapse to near-identical across the model? Map-set spread vs input-set
   spread, random / xavier / shuffled-pairing baselines, MLP Jacobian sensitivity, CKA.
5. **Summary** — fMRI vs EEG side by side.

> **Data-dependence:** MLP maps are *not* stored in the checkpoint. The checkpoint rebuilds
> the model; a forward pass over probe windows instantiates the maps. Configure paths via
> the env vars in §0. Run where the checkpoints + data live (the cluster).
>
> **fMRI needs an MLP-mode checkpoint** (`sheaf_mlp_maps=True`). A static fMRI run will trip
> the guard in `build_model_from_ckpt`. EEG LEMON arms are already MLP-mode.


## §0 · Config + imports

In [ ]:
import os, sys, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(42)

PROJECT_ROOT = os.environ.get(
    "BRAINDYN_ROOT", "~/project_<account>/<netid>/brainy/BrainDyn-SUMRY"
)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
import analysis.restriction_maps as rm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

# ── Checkpoints + data (override via env) ─────────────────────────────────────
# Each checkpoint MUST be an MLP-map run (sheaf_mlp_maps=True).
DATASETS = {
   # "fMRI_pernode": dict(kind="fmri", ckpt=os.environ.get("FMRI_CKPT", "checkpoints/fmri_full_pnc_short_pernode_ws_ft_hd16b_fold1.pt"),
    #             manifest=os.environ.get("FMRI_MANIFEST", "data/manifest_fmri.csv")),
    #"fMRI": dict(kind="fmri", ckpt=os.environ.get("FMRI_CKPT", "../../../shared/SUMRY_2026_BrainDyn/trained_models/fmri_linear_message_passing/pnc_linear_msgpass_direct_mlp_hd16_topk2000_seed42_fold1.pt"),
    #             manifest=os.environ.get("FMRI_MANIFEST", "data/manifest_fmri.csv")),
    #"EEG_ec":  dict(kind="eeg",  ckpt=os.environ.get("EEG_CKPT", "checkpoints/eeg_202sub_baseline_hd16_fold2.pt"),
    #             manifest=os.environ.get("LEMON_MANIFEST", "data/lemon_manifest.csv")),
    #"EEG_random": dict(kind="eeg",  ckpt=os.environ.get("EEG_CKPT", "checkpoints/eeg_202sub_random_hd16_fold2.pt"),
     #            manifest=os.environ.get("LEMON_MANIFEST", "data/lemon_manifest.csv")),
    #"EEG_eo": dict(kind="eeg",  ckpt=os.environ.get("EEG_CKPT", "checkpoints/eeg_194sub_baseline_hd16_EO_fold2.pt"),
     #            manifest=os.environ.get("LEMON_MANIFEST", "data/lemon_manifest.csv")),
    #"EEG_ar": dict(kind="eeg",  ckpt=os.environ.get("EEG_CKPT", "checkpoints/eeg_arseg_EC_baseline_hd16_fold1.pt"),
    #             manifest=os.environ.get("LEMON_MANIFEST", "data/lemon_manifest.csv")),
    #"EEG_ar_pe": dict(kind="eeg",  ckpt=os.environ.get("EEG_CKPT", "checkpoints/eeg_arseg_EC_LapPE_hd16_fold1.pt"),
     #            manifest=os.environ.get("LEMON_MANIFEST", "data/lemon_manifest.csv")),   
    "EEG_best": dict(kind="eeg",  ckpt=os.environ.get("EEG_CKPT", "checkpoints/paper_eeg/braindyn_eeg_short_granger_fold4.pt"),
                 manifest=os.environ.get("LEMON_MANIFEST", "data/lemon_manifest.csv")),
    #"EEG_worst": dict(kind="eeg",  ckpt=os.environ.get("EEG_CKPT", "checkpoints/paper_eeg/braindyn_eeg_short_granger_fold3.pt"),
      #           manifest=os.environ.get("LEMON_MANIFEST", "data/lemon_manifest.csv")), 
    #"fMRI_best": dict(kind="fmri", ckpt=os.environ.get("FMRI_CKPT", "checkpoints/paper_fmri/braindyn_fmri_short_granger_fold4.pt"),
      #           manifest=os.environ.get("FMRI_MANIFEST", "data/manifest_fmri.csv")),
    #"fMRI_worst": dict(kind="fmri", ckpt=os.environ.get("FMRI_CKPT", "checkpoints/paper_fmri/braindyn_fmri_short_granger_fold3.pt"),
     #            manifest=os.environ.get("FMRI_MANIFEST", "data/manifest_fmri.csv")),    
    "EEG_bestpe": dict(kind="eeg",  ckpt=os.environ.get("EEG_CKPT", "checkpoints/paper_eeg/braindyn_eeg_short_pe_fold4.pt"),
                 manifest=os.environ.get("LEMON_MANIFEST", "data/lemon_manifest.csv")),
}

DATASETS = {k: v for k, v in DATASETS.items() if v["ckpt"]}

MAX_BATCHES = int(os.environ.get("MAX_BATCHES", "4"))   # probe batches to pool
BATCH_SIZE  = int(os.environ.get("BATCH_SIZE", "64"))
SUBSAMPLE   = int(os.environ.get("SUBSAMPLE", "1500"))  # cap for O(P^2) / Jacobian ops
CONDITION   = os.environ.get("CONDITION", "EC")

print("configured datasets:", list(DATASETS) or "(none — set FMRI_CKPT / EEG_CKPT)")
print("device:", device)

# Optional extra checkpoints to compare as "conditions" (same node space; e.g. EC vs EO)
# for the cross-condition Gram in §4c. COND_CKPTS is comma-separated; COND_MANIFESTS pairs
# with it (or a single manifest reused); COND_KIND is fmri|eeg.
_cc = [p for p in os.environ.get("COND_CKPTS", "").split(",") if p]
_cm = [p for p in os.environ.get("COND_MANIFESTS", "").split(",") if p]
_cn = [p for p in os.environ.get("COND_NAMES", "").split(",") if p]
COND_KIND = os.environ.get("COND_KIND", "eeg")
CONDITIONS = [dict(name=(_cn[i] if i < len(_cn) else f"cond{i+1}"), ckpt=ck, kind=COND_KIND,
                   manifest=(_cm[i] if i < len(_cm) else (_cm[0] if _cm else "")))
              for i, ck in enumerate(_cc)]
print("conditions (cross-condition Gram):", [c["name"] for c in CONDITIONS] or "(none)")

## §1 · Load checkpoints & harvest maps

Rebuild each model (asserting MLP mode), take a probe loader, and pool the map population
across `MAX_BATCHES` batches into `(S, E, H, d_e)`. Everything downstream reads from `HARVEST`.

In [ ]:
from data.rbc_dataset import make_dataloaders as make_fmri_loaders
from data.lemon_dataset import make_lemon_dataloaders


def build_loaders(dc, cfg):
    """Build the full split dict of DataLoaders for the dataset."""
    g = cfg.get
    x, y, stride = g("x", 30), g("y", 10), g("stride", 10)
    if dc["kind"] == "fmri":
        return make_fmri_loaders(
            dc["manifest"], x=x, y=y, stride=stride, batch_size=BATCH_SIZE,
            num_workers=0, pin_memory=False, norm_mode=g("norm_mode", "train_global"),
        )
    return make_lemon_dataloaders(
        manifest_csv=dc["manifest"], x=x, y=y, stride=stride, batch_size=BATCH_SIZE,
        num_workers=0, condition=g("condition", CONDITION),
        max_subjects=g("max_subjects"), max_windows_per_subject=g("max_windows_per_subject"),
        seed=g("lemon_data_seed", 42), pin_memory=False, norm_mode=g("norm_mode", "train_global"),
    )


def pick_split(loaders, prefer=("test", "val", "train")):
    """Return (loader, split_name) for the first available split."""
    for split in prefer:
        if loaders.get(split) is not None and len(loaders[split].dataset) > 0:
            return loaders[split], split
    raise RuntimeError("no usable split in loaders")


HARVEST = {}
for name, dc in DATASETS.items():
    ckpt = torch.load(dc["ckpt"], map_location="cpu", weights_only=False)
    ei = ckpt["edge_index"].to(device)
    N = rm.num_nodes_from_ckpt(ckpt)
    model = rm.build_model_from_ckpt(ckpt, ei, device=device, num_nodes=N, ckpt_name=dc["ckpt"])
    loaders = build_loaders(dc, ckpt["config"])
    loader, split = pick_split(loaders)
    H = rm.harvest_over_loader(model, loader, ei, norm_stats=ckpt.get("norm_stats"),
                               max_batches=MAX_BATCHES, device=device)
    HARVEST[name] = dict(H=H, model=model, ckpt=ckpt, ei=ei, N=N, E=int(ei.shape[1]),
                         loaders=loaders, split=split,
                         dt=float(ckpt["config"].get("dt", 1.0) or 1.0))
    R = H["rho_src"]
    print(f"{name:5s}  N={N:4d}  E={ei.shape[1]:5d}  maps {R.shape}  split={split}  "
          f"(H={R.shape[-2]}, d_e={R.shape[-1]}, square={R.shape[-2] == R.shape[-1]})")
assert HARVEST, "No datasets harvested — set FMRI_CKPT / EEG_CKPT."
n = len(HARVEST)                       # reused for subplot layouts in later sections
summary = {name: {} for name in HARVEST}   # per-dataset scalars, filled across sections

## §1b · Gauge-invariant map metrics (normalized SVD + diffusion gain)

With `--sheaf_norm sym` the operator is `D^{-1/2} L_F D^{-1/2}`: its spectrum is bounded to
`[0, 2]` **regardless of the map scale**, so the old stability ceiling that used to pin `‖ρ‖`
small is gone. The overall map scale is now a **gauge** — `ρ → cρ` is absorbed by the learnable
`diffusion_gain` (`gl.diffusion_gain()`) with identical model output. So raw `‖ρ‖` and its
comparison to `isometry = 1` are no longer meaningful, and "every σ_k ≥ init" is partly a scale
artifact.

This section reports only quantities that survive the gauge:
- **Normalized SVD spectrum** — each map's singular values divided by its own `σ₁` (pure *shape*),
  with the init/random null **rescaled to matched scale** so a scale offset cannot masquerade as
  reshaping.
- **Diffusion gain** — the *actual* per-step update each edge delivers,
  `eff_step · ‖message‖` with `eff_step = diffusion_step · diffusion_gain()`. This is the
  gauge-invariant physical displacement (`sheaf_h − h_t`, what the ODE head consumes), decomposed
  per edge — the honest replacement for raw `‖ρ‖` and for the old single "relative sheaf effect".

Toggles: `GAIN_MODE` = `absolute` (default, `eff_step·‖msg‖`) | `relative` (÷ receiving-node `‖h‖`);
`INIT_RESCALE` = how the null is scale-matched in the raw panel.

In [ ]:
# §1b helpers — self-contained scalp drawing + gauge-invariant toggles (no dependency on §8).
from matplotlib.collections import LineCollection as _LC
from pathlib import Path as _Path
from data.lemon_dataset import montage_positions as _montage_positions

GAIN_MODE    = os.environ.get("GAIN_MODE", "absolute")       # "absolute" = eff_step*||msg||  |  "relative" = / ||h_recv||
INIT_RESCALE = os.environ.get("INIT_RESCALE", "mean_sigma")  # raw-panel null scaling: "mean_sigma" | "none"
SNAP_START   = os.environ.get("GAIN_SNAP_START", "")         # "" -> center the 6 adjacent windows; else int

def _gi_scalp_xy(pos3d):
    # azimuthal-equidistant flattening (nose up), same as §8 — low electrodes fan out correctly.
    p = np.asarray(pos3d, float)
    q = p - np.array([0.0, 0.0, p[:, 2].min()])
    r = np.linalg.norm(q, axis=1); r[r == 0] = 1.0
    theta = np.arccos(np.clip(q[:, 2] / r, -1.0, 1.0))
    phi = np.arctan2(q[:, 1], q[:, 0])
    return np.c_[theta * np.cos(phi), theta * np.sin(phi)]

def _gi_montage(rec):
    ds = rec["loaders"]["train"].dataset
    ch = list(ds.channels)
    pj = _Path(ds._samples[0][0]).parent / "montage_positions.json"
    pos3d = _montage_positions(ch, positions_json=str(pj) if pj.exists() else None)
    return _gi_scalp_xy(pos3d)

def _gi_head(ax, pos2d):
    r = np.abs(pos2d).max() * 1.15; th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(r * np.cos(th), r * np.sin(th), color="k", lw=1)
    ax.plot([-0.09 * r, 0, 0.09 * r], [r * 0.99, r * 1.12, r * 0.99], color="k", lw=1)

def _gi_draw_edges(ax, ei_np, pos2d, edge_vals, title, cmap="viridis", vmin=0.0, vmax=None):
    _gi_head(ax, pos2d)
    v = np.asarray(edge_vals, float)
    vmax = np.nanpercentile(v, 99) if vmax is None else vmax
    segs = [[(pos2d[s, 0], pos2d[s, 1]), (pos2d[d, 0], pos2d[d, 1])] for s, d in zip(ei_np[0], ei_np[1])]
    lc = _LC(segs, cmap=cmap, norm=plt.Normalize(vmin, vmax), lw=1.6, alpha=0.85)
    lc.set_array(v); ax.add_collection(lc)
    ax.scatter(pos2d[:, 0], pos2d[:, 1], s=16, c="k", zorder=3)
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.axis("off")
    return lc

def _gi_draw_half_edges(ax, src, dst, pos2d, c_to_src, c_to_dst, title, cmap="magma", vmax=None):
    _gi_head(ax, pos2d)
    segs, vals = [], []
    for e in range(len(src)):
        ps, pd = pos2d[src[e]], pos2d[dst[e]]; mid = 0.5 * (ps + pd)
        segs.append([tuple(ps), tuple(mid)]); vals.append(c_to_src[e])
        segs.append([tuple(pd), tuple(mid)]); vals.append(c_to_dst[e])
    vals = np.asarray(vals, float)
    vmax = np.nanpercentile(vals, 99) if vmax is None else vmax
    lc = _LC(segs, cmap=cmap, norm=plt.Normalize(0, vmax), lw=1.8, alpha=0.85)
    lc.set_array(vals); ax.add_collection(lc)
    ax.scatter(pos2d[:, 0], pos2d[:, 1], s=14, c="k", zorder=3)
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.axis("off")
    return lc

def harvest_diffusion_gain(rec, gain_mode=None):
    """Per-(window, edge) diffusion gain = eff_step*||msg|| each half-edge delivers.

    eff_step = diffusion_step * diffusion_gain() (the effective step the model actually applies; the
    shared helper only multiplies by diffusion_step, so we fold in the learned gain here). Single pass
    so the src/dst ||h|| used by the "relative" mode is aligned to the same windows.
    """
    from main import batch_to_model_tensors
    gain_mode = GAIN_MODE if gain_mode is None else gain_mode
    model, ei = rec["model"], rec["ei"]
    gl = model.dynamics.graph_laplacian
    g = float(gl.diffusion_gain()) if hasattr(gl, "diffusion_gain") else 1.0
    src = ei[0].detach().cpu().numpy(); dst = ei[1].detach().cpu().numpy()
    loader, _ = pick_split(rec["loaders"])
    cs_l, cd_l, hs_l, hd_l = [], [], [], []
    for i, batch in enumerate(loader):
        if i >= MAX_BATCHES:
            break
        x, _ = batch_to_model_tensors(batch, device, norm_stats=rec["ckpt"].get("norm_stats"))
        ec = rm.sheaf_edge_contributions(model, x, ei, round_idx=0)
        cs_l.append((g * ec["c_to_src"]).detach().cpu().numpy())
        cd_l.append((g * ec["c_to_dst"]).detach().cpu().numpy())
        if gain_mode == "relative":
            sd = rm.sheaf_step_decomposition(model, x, ei, round_idx=0)
            hn = sd["h_norm"].detach().cpu().numpy()          # (B, N)
            hs_l.append(hn[:, src]); hd_l.append(hn[:, dst])  # receiving-node ||h|| per half-edge
    cs = np.concatenate(cs_l, 0); cd = np.concatenate(cd_l, 0)  # (S, E)
    # Fold in the OUTPUT degree normalization (D^{-1/2} at the receiving node) that forward applies
    # after aggregation but sheaf_edge_contributions omits, so each value is this edge's TRUE
    # contribution to eff_step*(L_F h) at that node (ones for sheaf_norm="none").
    sc = gl._node_scale(ei, ei.device, torch.float32).detach().cpu().numpy()
    cs = cs * sc[src]; cd = cd * sc[dst]
    if gain_mode == "relative":
        cs = cs / (np.concatenate(hs_l, 0) + 1e-12)
        cd = cd / (np.concatenate(hd_l, 0) + 1e-12)
    return dict(c_to_src=cs, c_to_dst=cd, src=src, dst=dst, gain=g, mode=gain_mode)

_EEG1 = next((k for k, v in DATASETS.items() if v["kind"] == "eeg" and k in HARVEST), None)
print("§1b helpers ready | GAIN_MODE:", GAIN_MODE, "| INIT_RESCALE:", INIT_RESCALE,
      "| EEG for scalp:", _EEG1 or "(none)")

In [ ]:
# Normalized SVD spectrum — gauge-invariant map SHAPE, with the random-init null rescaled to match.
# Left: each map's singular values / its own sigma_1 (pure reshaping; scale removed on BOTH curves).
# Right: raw sigma with the init null rescaled to the learned mean-sigma (INIT_RESCALE), so a scale
# offset alone cannot produce "every sigma_k >= init". Effective rank is scale-free either way.
_nr = len(HARVEST)
fig, axes = plt.subplots(_nr, 2, figsize=(12, 4.3 * _nr), squeeze=False)
C_L, C_I = "#1f77b4", "#9aa0a6"
for row, (name, rec) in enumerate(HARVEST.items()):
    R = np.asarray(rec["H"]["rho_src"], float); Hh, De = R.shape[-2], R.shape[-1]
    R2 = R.reshape(-1, Hh, De)
    init = rm.random_baseline((Hh, De), R2.shape[0], kind="xavier", seed=0)
    sv, isv = rm.svdvals(R2), rm.svdvals(init)                        # (P, r)
    er, ier = rm.effective_rank(R2).mean(), rm.effective_rank(init).mean()
    k = np.arange(1, sv.shape[1] + 1)

    svn = sv / (sv[:, :1] + 1e-12); isvn = isv / (isv[:, :1] + 1e-12)   # sigma_1-normalized
    a = axes[row, 0]
    a.fill_between(k, *np.percentile(svn, [25, 75], 0), color=C_L, alpha=0.18)
    a.plot(k, svn.mean(0), "-o", ms=4, lw=2, color=C_L, label="Learned")
    a.fill_between(k, *np.percentile(isvn, [25, 75], 0), color=C_I, alpha=0.18)
    a.plot(k, isvn.mean(0), "--s", ms=3, lw=1.6, color=C_I, label="Init (random)")
    a.set_xlabel("singular index k"); a.set_ylabel(r"$\sigma_k/\sigma_1$"); a.set_ylim(0, 1.02)
    a.set_title(f"{name} — normalized spectrum (shape)   eff.rank {er:.2f} vs {ier:.2f}"); a.legend()

    isv_r = isv * (sv.mean() / (isv.mean() + 1e-12)) if INIT_RESCALE == "mean_sigma" else isv.copy()
    lbl = "Init (scale-matched)" if INIT_RESCALE == "mean_sigma" else "Init (random)"
    b = axes[row, 1]
    b.fill_between(k, *np.percentile(sv, [25, 75], 0), color=C_L, alpha=0.18)
    b.plot(k, sv.mean(0), "-o", ms=4, lw=2, color=C_L, label="Learned")
    b.fill_between(k, *np.percentile(isv_r, [25, 75], 0), color=C_I, alpha=0.18)
    b.plot(k, isv_r.mean(0), "--s", ms=3, lw=1.6, color=C_I, label=lbl)
    b.set_xlabel("singular index k"); b.set_ylabel(r"$\sigma_k$"); b.set_ylim(bottom=0)
    b.set_title(f"{name} — raw spectrum (INIT_RESCALE={INIT_RESCALE})"); b.legend()
    summary.setdefault(name, {})["effrank_norm"] = float(er)
fig.suptitle("Normalized restriction-map spectra — shape vs scale-matched init", fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.99]); plt.show()

In [ ]:
# Diffusion gain distribution — the per-half-edge update the sheaf step actually delivers.
# Pools both half-edges over all probe windows. Gauge-invariant, per-edge decomposition of the total
# diffusion displacement (the old single "relative sheaf effect" number, done right & split by edge).
DIFF_GAIN = {name: harvest_diffusion_gain(rec) for name, rec in HARVEST.items()}
unit = ("eff_step·‖msg‖" if GAIN_MODE == "absolute" else "eff_step·‖msg‖ / ‖h_recv‖")
fig, ax = plt.subplots(figsize=(7.6, 4.4))
for name, dg in DIFF_GAIN.items():
    vals = np.concatenate([dg["c_to_src"].ravel(), dg["c_to_dst"].ravel()])
    ax.hist(vals, bins=70, alpha=0.5, density=True,
            label=f"{name}  (mean {vals.mean():.3g}, learned g={dg['gain']:.2g})")
    summary.setdefault(name, {})["diff_gain_mean"] = float(vals.mean())
    summary[name]["diff_gain_median"] = float(np.median(vals))
ax.set_xlabel(f"diffusion gain per half-edge   [{unit}]  (GAIN_MODE={GAIN_MODE})")
ax.set_ylabel("density"); ax.set_title("Per-edge diffusion gain — what each edge adds to the encoding")
ax.legend(fontsize=8); fig.tight_layout(); plt.show()
for name, dg in DIFF_GAIN.items():
    print(f"{name:10s}  half-edges/window={2*len(dg['src'])}  learned diffusion_gain g={dg['gain']:.3g}  "
          f"mean={summary[name]['diff_gain_mean']:.3g}  median={summary[name]['diff_gain_median']:.3g}")

In [ ]:
# Diffusion gain on the scalp — averaged across each edge's two halves. Left: window-averaged field;
# right grid: six ADJACENT windows (shared colour scale) so the time variation of the diffusion is visible.
if _EEG1 is None:
    print("No EEG dataset — skipping the edge-averaged diffusion-gain scalp.")
else:
    rec = HARVEST[_EEG1]; dg = DIFF_GAIN[_EEG1]
    ei_np = rec["ei"].detach().cpu().numpy(); pos2d = _gi_montage(rec)
    edge_win = 0.5 * (dg["c_to_src"] + dg["c_to_dst"])              # (S, E)  averaged across the edge
    S = edge_win.shape[0]; kk = min(6, S)
    start = (max(0, S // 2 - kk // 2) if SNAP_START == "" else int(SNAP_START))
    start = min(start, max(0, S - kk)); snap = np.arange(start, start + kk)
    vmax = float(np.nanpercentile(edge_win, 99))
    fig = plt.figure(figsize=(16, 8)); gs = fig.add_gridspec(2, 5)
    a0 = fig.add_subplot(gs[:, 0:2])
    lc = _gi_draw_edges(a0, ei_np, pos2d, edge_win.mean(0), f"{_EEG1}: window-averaged", vmax=vmax)
    fig.colorbar(lc, ax=a0, fraction=0.046, label=f"diffusion gain ({dg['mode']})")
    for j, w in enumerate(snap):
        ax = fig.add_subplot(gs[j // 3, 2 + j % 3])
        _gi_draw_edges(ax, ei_np, pos2d, edge_win[w], f"window {w}", vmax=vmax)
    fig.suptitle(f"{_EEG1} · edge-averaged diffusion gain across windows "
                 f"(GAIN_MODE={GAIN_MODE}, g={dg['gain']:.2g})", fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

In [ ]:
# Diffusion gain on the scalp — each edge SPLIT IN TWO (src-half vs dst-half; the endpoint maps differ,
# so the two contributions differ). Left: window-averaged; right grid: six adjacent windows (shared scale).
if _EEG1 is None:
    print("No EEG dataset — skipping the split-edge diffusion-gain scalp.")
else:
    rec = HARVEST[_EEG1]; dg = DIFF_GAIN[_EEG1]
    pos2d = _gi_montage(rec); src, dst = dg["src"], dg["dst"]
    cs, cd = dg["c_to_src"], dg["c_to_dst"]                         # (S, E)
    S = cs.shape[0]; kk = min(6, S)
    start = (max(0, S // 2 - kk // 2) if SNAP_START == "" else int(SNAP_START))
    start = min(start, max(0, S - kk)); snap = np.arange(start, start + kk)
    vmax = float(np.nanpercentile(np.concatenate([cs.ravel(), cd.ravel()]), 99))
    fig = plt.figure(figsize=(16, 8)); gs = fig.add_gridspec(2, 5)
    a0 = fig.add_subplot(gs[:, 0:2])
    lc = _gi_draw_half_edges(a0, src, dst, pos2d, cs.mean(0), cd.mean(0),
                             f"{_EEG1}: window-averaged (split)", vmax=vmax)
    fig.colorbar(lc, ax=a0, fraction=0.046, label="half-edge diffusion gain")
    for j, w in enumerate(snap):
        ax = fig.add_subplot(gs[j // 3, 2 + j % 3])
        _gi_draw_half_edges(ax, src, dst, pos2d, cs[w], cd[w], f"window {w}", vmax=vmax)
    fig.suptitle(f"{_EEG1} · split-edge diffusion gain across windows "
                 f"(GAIN_MODE={GAIN_MODE}, g={dg['gain']:.2g})", fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()
    print(f"{_EEG1}: mean src-half={cs.mean():.3g}  dst-half={cd.mean():.3g}  "
          f"asymmetry={abs(cs.mean()-cd.mean())/(cs.mean()+cd.mean()+1e-12):.2%}")

## §2 · Norms — average & distribution

Norms are **normalized so an isometry = 1** (`NORM="isometry"` → divide by `√min(H,d_e)`;
the identity, an orthogonal map, or any rectangular map with orthonormal columns all read
1.0). `<1` is contractive, `>1` expansive. Set `NORM="rms"` to reproduce
`analyze_restriction_maps.py` (`÷√(H·d_e)`, a per-entry RMS — note that does *not* equal 1
for the identity), or `NORM=None` for the raw operator norm.

Each map's norm varies for two separable reasons, plotted side by side:
- **every map** (full `S·E` population) — total spread.
- **per-edge mean over windows** — *between-edge* / structural variance (the only variance
  a static map could have: which edges carry larger maps).
- **per-edge std across windows** — *within-edge* / state-dependence variance (non-zero only
  because the MLP maps are data-dependent: how much one edge's map moves with the input).

`total ≈ between + within`, so `between_frac` says which source dominates.

In [ ]:
NORM = "isometry"   # 1.0 == identity / orthogonal / isometry.  "rms" matches analyze_restriction_maps.py;  None = raw ||ρ||_F

fig, axes = plt.subplots(len(HARVEST), 3, figsize=(15, 4.0 * len(HARVEST)), squeeze=False)
for row, (name, rec) in enumerate(HARVEST.items()):
    st = rm.edge_norm_stats(rec["H"]["rho_src"], normalize=NORM)
    a0, a1, a2 = axes[row]
    # (a) full population — total spread
    a0.hist(st["per_map"], bins=60, color="#4878CF", alpha=0.85)
    if NORM == "isometry":
        a0.axvline(1.0, color="k", ls=":", lw=1.2, label="isometry = 1")
    a0.axvline(st["per_map"].mean(), color="crimson", ls="--", label=f"mean {st['per_map'].mean():.3f}")
    a0.set_title(f"{name} · every map  (n={st['per_map'].size})")
    a0.set_xlabel(f"‖ρ‖ [{NORM}]"); a0.set_ylabel("count"); a0.legend()
    # (b) per-edge mean over windows — between-edge (structural)
    a1.hist(st["edge_mean"], bins=40, color="#6ACC65", alpha=0.85)
    if NORM == "isometry":
        a1.axvline(1.0, color="k", ls=":", lw=1.2)
    a1.set_title(f"{name} · per-edge mean  (between-var={st['between_var']:.2e})")
    a1.set_xlabel("window-averaged ‖ρ‖"); a1.set_ylabel("edges")
    # (c) per-edge std across windows — within-edge (state dependence)
    a2.hist(st["edge_std"], bins=40, color="#B47CC7", alpha=0.85)
    a2.set_title(f"{name} · per-edge std / windows  (within-var={st['within_var']:.2e})")
    a2.set_xlabel("std of ‖ρ‖ across windows"); a2.set_ylabel("edges")
fig.tight_layout(); plt.show()

print(f"Norm variance decomposition (normalize={NORM!r}; 1.0 = identity/orthogonal):")
for name, rec in HARVEST.items():
    st = rm.edge_norm_stats(rec["H"]["rho_src"], normalize=NORM)
    print(f"  {name:5s}  mean={st['per_map'].mean():.3f}  between-edge={st['between_var']:.3e}  "
          f"within-edge(windows)={st['within_var']:.3e}  total={st['total_var']:.3e}  "
          f"between_frac={st['between_frac']:.2f}")

## §3 · Effective rank

Participation-ratio effective rank per map (1 = rank-1, `min(H,d_e)` = flat spectrum) and
the mean singular spectrum (± std). Low effective rank ⇒ each map collapses signal onto a
few directions.

In [ ]:
fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 3.8), squeeze=False)
for ax, (name, rec) in zip(axes[0], HARVEST.items()):
    R = rec["H"]["rho_src"]
    er = rm.effective_rank(R, "participation")
    full = min(R.shape[-2], R.shape[-1])
    ax.hist(er, bins=40, color="#6ACC65", alpha=0.85)
    ax.axvline(full, color="k", ls="--", label=f"full rank {full}")
    ax.axvline(er.mean(), color="crimson", ls="--", label=f"mean {er.mean():.2f}")
    ax.set_title(f"{name}  effective rank"); ax.set_xlabel("participation ratio")
    ax.set_ylabel("count"); ax.legend()
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4))
for name, rec in HARVEST.items():
    s = rm.svdvals(rec["H"]["rho_src"])
    m, sd = s.mean(0), s.std(0)
    xs = np.arange(1, len(m) + 1)
    ax.plot(xs, m, marker="o", label=name)
    ax.fill_between(xs, m - sd, m + sd, alpha=0.2)
ax.set_xlabel("singular-value index"); ax.set_ylabel("σ (mean ± std)")
ax.set_title("Restriction-map singular spectrum"); ax.legend(); plt.show()

## §4 · Polar decomposition — rotation vs scaling

`ρ = Q P` (SVD-based): `Q = U Vᵀ` the nearest partial isometry (**rotation/reflection**),
`P = V Σ Vᵀ` the SPD **stretch** (scaling). `rotation_frac = rotation_dev / (rotation_dev +
stretch_dev)` says how much of each map's deviation from identity is rotation vs scaling.

In [ ]:
fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 3.8), squeeze=False)
for ax, (name, rec) in zip(axes[0], HARVEST.items()):
    pol = rm.svd_polar(rec["H"]["rho_src"])
    ax.hist(pol["rotation_frac"], bins=40, color="#B47CC7", alpha=0.85)
    ax.axvline(pol["rotation_frac"].mean(), color="crimson", ls="--",
               label=f"mean {pol['rotation_frac'].mean():.3f}")
    ax.axvline(0.5, color="k", ls=":", lw=1)
    ax.set_title(f"{name}  rotation fraction"); ax.set_xlabel("rotation / (rotation+scaling)")
    ax.set_ylabel("count"); ax.legend()
fig.tight_layout(); plt.show()

fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.2), squeeze=False)
for ax, (name, rec) in zip(axes[0], HARVEST.items()):
    pol = rm.svd_polar(rec["H"]["rho_src"])
    ax.scatter(pol["stretch_dev"], pol["rotation_dev"], s=6, alpha=0.3, color="#4878CF")
    lim = max(pol["stretch_dev"].max(), pol["rotation_dev"].max())
    ax.plot([0, lim], [0, lim], "k:", lw=1)
    ax.set_xlabel("scaling  ‖P − I‖_F"); ax.set_ylabel("rotation  ‖Q − I‖_F")
    ax.set_title(f"{name}  (above line ⇒ rotation-dominated)")
fig.tight_layout(); plt.show()

for name, rec in HARVEST.items():
    pol = rm.svd_polar(rec["H"]["rho_src"])
    print(f"  {name:5s}  rotation_frac mean={pol['rotation_frac'].mean():.3f}"
          f"  scale_dev mean={pol['scale_dev'].mean():.3f}"
          f"  total_dev mean={pol['total_dev'].mean():.3f}")

### §4b · Orthogonal component — absolute magnitude, not just direction

`rotation_frac` (§4) is a *direction-of-deviation* ratio: "of how far R sits from I, what
fraction is rotation vs scaling". It is **not** how much of an orthogonal matrix R is. For a
contractive map (`‖R‖ < 1`) the two diverge sharply. `orthogonal_content` gives the absolute
picture via `R ≈ s·Q + residual`:
- **`ortho_scale` `s = mean(σ)`** — "R acts like *s* × an orthogonal matrix" (= 1 for a true
  isometry). So `rotation_frac ≈ 0.65` with `ortho_scale ≈ 0.2` means: *0.2 of an orthogonal
  matrix, and of that small deviation, 65% is rotational* — exactly your hunch.
- **`ortho_energy_frac`** — share of R's energy in the isotropic `s·Q` part (1 ⇔ perfectly
  `s·orthogonal`).

In [ ]:
fig, axes = plt.subplots(len(HARVEST), 2, figsize=(11, 4.1 * len(HARVEST)), squeeze=False)
for row, (name, rec) in enumerate(HARVEST.items()):
    R = rec["H"]["rho_src"]
    oc = rm.orthogonal_content(R); pol = rm.svd_polar(R)
    a0, a1 = axes[row]
    a0.hist(oc["orthogonal_scale"], bins=50, color="#B47CC7", alpha=0.85)
    a0.axvline(1.0, color="k", ls=":", lw=1.2, label="orthogonal = 1")
    a0.axvline(oc["orthogonal_scale"].mean(), color="crimson", ls="--", label=f"mean {oc['orthogonal_scale'].mean():.3f}")
    a0.set_title(f"{name} · orthogonal_scale  (× an orthogonal matrix)")
    a0.set_xlabel("s = mean singular value"); a0.set_ylabel("count"); a0.legend(fontsize=8)
    a1.scatter(pol["rotation_frac"], oc["orthogonal_scale"], s=6, alpha=0.3, color="#0f766e")
    a1.axhline(1.0, color="k", ls=":", lw=1)
    a1.set_xlabel("rotation_frac (direction of deviation)")
    a1.set_ylabel("orthogonal_scale (absolute magnitude)")
    a1.set_title(f"{name} · direction vs magnitude")
    summary[name]["ortho_scale"] = float(oc["orthogonal_scale"].mean())
    summary[name]["ortho_energy_frac"] = float(oc["isometric_energy_frac"].mean())
fig.tight_layout(); plt.show()

print("Reconciling rotation_frac (direction) with the absolute orthogonal scale:")
for name, rec in HARVEST.items():
    R = rec["H"]["rho_src"]; oc = rm.orthogonal_content(R); pol = rm.svd_polar(R)
    print(f"  {name:5s}  rotation_frac={pol['rotation_frac'].mean():.3f} (of the deviation)  "
          f"orthogonal_scale={oc['orthogonal_scale'].mean():.3f} (x orthogonal)  "
          f"energy_frac={oc['isometric_energy_frac'].mean():.3f}")

### §4c · Cross-condition Gram — do different conditions recover the same maps?

Given >=2 checkpoints on the **same node space** (`COND_CKPTS`, e.g. EC vs EO), compare each
edge's window-averaged **orthogonal part** across conditions, over the edges the two share.
The cosine **Gram** matrix has same-edge similarity on its diagonal and the shuffled baseline
off-diagonal — a strong diagonal (`diag_mean >> offdiag_mean`) means the two conditions
converge to the *same* per-edge map, not merely similar-looking maps globally.

In [ ]:
def _cond_edge_ortho(ckpt_path, kind, manifest):
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    ei = ck["edge_index"].to(device); N = rm.num_nodes_from_ckpt(ck)
    model = rm.build_model_from_ckpt(ck, ei, device=device, num_nodes=N, ckpt_name=ckpt_path)
    loaders = build_loaders(dict(kind=kind, manifest=manifest), ck["config"])
    loader, _ = pick_split(loaders)
    Hh = rm.harvest_over_loader(model, loader, ei, norm_stats=ck.get("norm_stats"),
                                max_batches=MAX_BATCHES, device=device)
    Rmean = Hh["rho_src"].mean(0)                                  # (E, H, d_e)
    pairs = np.stack([ei[0].cpu().numpy(), ei[1].cpu().numpy()], 1)
    return Rmean, pairs

if len(CONDITIONS) >= 2:
    a, b = CONDITIONS[0], CONDITIONS[1]
    Ra_full, pa = _cond_edge_ortho(a["ckpt"], a["kind"], a["manifest"])
    Rb_full, pb = _cond_edge_ortho(b["ckpt"], b["kind"], b["manifest"])
    Ra, Rb, common = rm.align_edge_maps(pa, Ra_full, pb, Rb_full)
    K = len(common)
    if K < 3:
        print(f"only {K} shared edges between {a['name']} and {b['name']}.")
    else:
        g = rm.cross_condition_gram(rm.polar_orthogonal(Ra), rm.polar_orthogonal(Rb))
        fig, ax = plt.subplots(1, 2, figsize=(12, 5))
        im = ax[0].imshow(g["gram"], cmap="magma", vmin=-1, vmax=1)
        ax[0].set_title(f"cosine Gram: {a['name']} vs {b['name']}  (K={K} shared edges)")
        ax[0].set_xlabel(f"{b['name']} edge"); ax[0].set_ylabel(f"{a['name']} edge")
        fig.colorbar(im, ax=ax[0], fraction=0.046)
        off = g["gram"][~np.eye(K, dtype=bool)] if K > 1 else np.array([np.nan])
        ax[1].hist(off, bins=40, range=(-1, 1), alpha=0.5, color="#64748b", label=f"different edge ({g['offdiag_mean']:.3f})")
        ax[1].hist(g["diag"], bins=40, range=(-1, 1), alpha=0.8, color="#2563eb", label=f"same edge ({g['diag_mean']:.3f})")
        ax[1].set_xlabel("cosine of orthogonal parts"); ax[1].set_ylabel("count")
        ax[1].set_title("same-edge vs shuffled"); ax[1].legend()
        fig.tight_layout(); plt.show()
        verdict = "SAME maps recovered" if g["separation"] > 0.1 else "not clearly aligned"
        print(f"{a['name']} vs {b['name']}: {K} shared edges | same-edge cos={g['diag_mean']:.3f} "
              f"vs shuffled={g['offdiag_mean']:.3f}  ->  {verdict}")
else:
    print("Set COND_CKPTS to >=2 comma-separated checkpoints on the same node space "
          "(e.g. EC,EO) to run the cross-condition Gram.")

### §4d · Smoothness of the orthogonal field vs random

Is the learned rotation field a *coherent global geometry* or per-edge noise? For each dataset
we measure the mean distance between the orthogonal parts of edges that **share a node**
(adjacent edges), against a random-orthogonal field of the same size. Learned `<<` random ⇒ the
model found a smooth, spatially-structured field of rotations.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
labels, learned, rand = [], [], []
for name, rec in HARVEST.items():
    R = rec["H"]["rho_src"]; ei = rec["ei"].detach().cpu().numpy()
    Qv = rm.polar_orthogonal(R.mean(0))                              # (E, H, d_e) orthogonal part
    pairs = np.stack([ei[0], ei[1]], 1)
    sm = rm.map_field_smoothness(Qv, pairs, n_null=10)
    labels.append(name); learned.append(sm["learned"]); rand.append(sm["null"])
    summary[name]["ortho_smooth_ratio"] = float(sm["smoothness_ratio"])
x = np.arange(len(labels)); w = 0.38
ax.bar(x - w / 2, learned, w, label="learned", color="#0f766e")
ax.bar(x + w / 2, rand, w, label="shuffled null", color="#94a3b8")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("mean adjacent-edge distance (lower = smoother)")
ax.set_title("Smoothness of the orthogonal restriction-map field"); ax.legend()
fig.tight_layout(); plt.show()
for name, l, r in zip(labels, learned, rand):
    print(f"  {name:5s}  learned={l:.4f}  null={r:.4f}  ratio={l / (r + 1e-12):.3f}  "
          f"({'smoother than random' if l < r else 'no smoother'})")

## §5 · Discriminability — can the MLP distinguish near-collapsed embeddings?

The maps are a function of the LSTM embeddings, which are known to be very similar. So:
does the sheaf-learner **amplify** those tiny differences into distinct maps, or **wash
them out**? We compare the population spread of the maps against the spread of their inputs
and against three baselines, then quantify local sensitivity (Jacobian) and shared structure
(CKA).

- **map-set eff-rank vs input-set eff-rank** — dimensionality of the actual map cloud vs its
  inputs.
- **xavier / orthogonal baselines** — spread of unstructured random maps of the same shape.
- **shuffled-pairing** — recompute maps with `h_dst` permuted across edges; if spread is
  unchanged, pairing carries no information.
- **Jacobian norm** — local Lipschitz of the MLP at each operating point (amplify vs damp).
- **CKA(inputs, maps)** — how much map structure is inherited from the embeddings.

In [ ]:
for name, rec in HARVEST.items():   # summary dict initialized in §1, filled across sections
    H = rec["H"]; R = H["rho_src"]
    Hd, De = R.shape[-2], R.shape[-1]
    Xf = rm.flatten_maps(R)                                   # (P, H*d_e)
    hp = np.concatenate([H["h_src"], H["h_dst"]], -1).reshape(-1, 2 * Hd)  # (P, 2H)
    P = Xf.shape[0]

    map_spread = rm.set_effective_rank(Xf)
    in_spread  = rm.set_effective_rank(hp)
    xav_spread = rm.set_effective_rank(rm.flatten_maps(rm.random_baseline((Hd, De), P, "xavier")))
    ort_spread = rm.set_effective_rank(rm.flatten_maps(rm.random_baseline((Hd, De), P, "orthogonal")))

    # shuffled-pairing control: run the real MLP on mismatched (h_src, h_dst') pairs.
    # mlp_pair_inputs rebuilds the MLP input incl. the sheaf map-PE (cell B) in the model's
    # own order, so this is agnostic to whether the run used --sheaf_map_pe.
    mlp = rec["model"].dynamics.graph_laplacian.restriction_mlp
    perm = np.random.default_rng(0).permutation(P)
    with torch.no_grad():
        shuf_in = torch.tensor(rm.mlp_pair_inputs(H, shuffle=perm), device=device)
        shuf = mlp(shuf_in).reshape(-1, Hd, De).cpu().numpy()
    shuf_spread = rm.set_effective_rank(rm.flatten_maps(shuf))

    # heavy ops on a capped subsample (full MLP input incl. PE, so widths match cell B)
    mlp_in = rm.mlp_pair_inputs(H)
    idx = np.random.default_rng(0).choice(P, size=min(SUBSAMPLE, P), replace=False)
    jac = rm.mlp_jacobian_norms(rec["model"], torch.tensor(mlp_in[idx], dtype=torch.float32))
    cka = rm.linear_cka(hp, Xf)
    asym = rm.frob(R - H["rho_dst"])                          # src-vs-dst ordered asymmetry

    summary[name].update(
        map_spread=map_spread, in_spread=in_spread, xav_spread=xav_spread,
        ort_spread=ort_spread, shuf_spread=shuf_spread,
        jac_mean=float(jac.mean()), jac_std=float(jac.std()),
        cka=cka, asym_mean=float(asym.mean()),
        frob_mean=float(rm.frob(R).mean()),
        frob_iso_mean=float(rm.frob(R, "isometry").mean()),
        effrank_mean=float(rm.effective_rank(R).mean()),
        rot_frac=float(rm.svd_polar(R)["rotation_frac"].mean()),
        Xf=Xf, idx=idx,
    )
    print(f"{name}: map-spread={map_spread:.2f}  input-spread={in_spread:.2f}  "
          f"shuffled={shuf_spread:.2f}  xavier={xav_spread:.2f}  ortho={ort_spread:.2f}  "
          f"CKA={cka:.3f}  Jac={jac.mean():.3f}±{jac.std():.3f}")

In [ ]:
# Spread bar chart + pairwise-distance heatmap of the map cloud
fig, axes = plt.subplots(1, n + 1, figsize=(5.0 * (n + 1), 4.0), squeeze=False)
ax0 = axes[0][0]
labels = ["maps", "inputs", "shuffled", "xavier", "ortho"]
keys = ["map_spread", "in_spread", "shuf_spread", "xav_spread", "ort_spread"]
xpos = np.arange(len(labels)); w = 0.8 / max(len(summary), 1)
for j, (name, s) in enumerate(summary.items()):
    ax0.bar(xpos + j * w, [s[k] for k in keys], width=w, label=name)
ax0.set_xticks(xpos + w * (len(summary) - 1) / 2); ax0.set_xticklabels(labels, rotation=20)
ax0.set_ylabel("set effective rank"); ax0.set_title("Population spread vs baselines"); ax0.legend()

from scipy.spatial.distance import squareform, pdist
for ax, (name, s) in zip(axes[0][1:], summary.items()):
    sub = s["Xf"][s["idx"]]
    D = squareform(pdist(sub, "euclidean"))
    im = ax.imshow(D, cmap="viridis"); ax.set_title(f"{name}  pairwise map distance")
    ax.set_xticks([]); ax.set_yticks([]); fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout(); plt.show()

## §6 · LSTM embeddings on the graph

Two readouts of how the trained encoder behaves *on the edges*:
- **neighbor cosine** `cos(h_src, h_dst)` per `(window, edge)` — how similar the two
  endpoints' LSTM embeddings are. High + tight ⇒ near-collapsed neighbors (the regime that
  makes the sheaf hard to exploit). Shown as the full `(window, edge)` spread **and** the
  per-edge mean across windows.
- **relative sheaf change** `‖sheaf_h − h_t‖ / ‖h_t‖` per window — how much the graph
  diffusion actually moves the encoding. ~0 ⇒ inert graph; larger ⇒ the sheaf reshapes node
  states before the ODE. Both means feed the §9 summary.

In [ ]:
fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 3.8), squeeze=False)
for ax, (name, rec) in zip(axes[0], HARVEST.items()):
    cos = rm.neighbor_cosine(rec["H"]["h_src"], rec["H"]["h_dst"])   # (S, E)
    ax.hist(cos.reshape(-1), bins=60, color="#4878CF", alpha=0.55, label="all (window, edge)")
    ax.hist(cos.mean(0), bins=40, color="#dc2626", alpha=0.6, label="per-edge mean")
    ax.axvline(cos.mean(), color="k", ls="--", lw=1.2, label=f"mean {cos.mean():.3f}")
    ax.set_title(f"{name} · neighbor cosine (LSTM embeddings)")
    ax.set_xlabel("cos(h_src, h_dst)"); ax.set_ylabel("count"); ax.legend(fontsize=8)
    summary[name]["neighbor_cos"] = float(cos.mean())
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6.8, 4))
for name, rec in HARVEST.items():
    rel = rm.sheaf_relative_change(rec["H"]["sheaf_h"], rec["H"]["h_t"])   # (S,)
    ax.hist(rel, bins=40, alpha=0.6, label=f"{name} (mean {rel.mean():.3f})")
    summary[name]["sheaf_rel"] = float(rel.mean())
ax.set_xlabel("‖sheaf_h − h_t‖ / ‖h_t‖  per window"); ax.set_ylabel("count")
ax.set_title("Relative sheaf-diffusion effect on the encoding")
ax.legend(); fig.tight_layout(); plt.show()

print("Graph-signal metrics:")
for name in HARVEST:
    print(f"  {name:5s}  neighbor_cos={summary[name]['neighbor_cos']:.3f}  "
          f"sheaf_rel={summary[name]['sheaf_rel']:.3f}")

### §6 (cont.) · Neighbor cosine **with** the map positional encoding

The cell above uses raw LSTM embeddings, `cos(h_src, h_dst)`. But the restriction MLP is actually
fed `[h ‖ pe]` per endpoint (`model/sheaf.py` `_edge_pe`), so "how similar are neighbours" *to the
map-MLP* is the cosine in that augmented space. `pe_src`/`pe_dst` are harvested through the model's
own PE path, so this works for **both** `--sheaf_map_pe lappe` **and** `learned` (width 0 ⇒ no
map-PE). `PE_EQUAL_WEIGHT=False` shows it exactly as fed (PE weighted by its own norm); set it True
to unit-normalise `h` and `pe` first, which reveals the *potential* separation PE carries when its
small norm would otherwise be swamped by `h` (the MLP can learn that reweighting).

In [ ]:
# Neighbor cosine including the map-PE — one curve LSTM-only, one LSTM+PE. Works for lappe & learned.
PE_EQUAL_WEIGHT = False   # False: exactly as the MLP is fed. True: unit-norm h and pe before concat
                          # (PE gets equal weight -> its potential separation, not swamped by ‖h‖).

def _augment(h, pe):
    if PE_EQUAL_WEIGHT:
        h = h / (np.linalg.norm(h, axis=-1, keepdims=True) + 1e-12)
        pe = pe / (np.linalg.norm(pe, axis=-1, keepdims=True) + 1e-12)
    return np.concatenate([h, pe], axis=-1)

fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 3.8), squeeze=False)
for ax, (name, rec) in zip(axes[0], HARVEST.items()):
    H = rec["H"]
    gl = getattr(rec["model"].dynamics, "graph_laplacian", None)
    pe_mode = getattr(gl, "sheaf_map_pe", "none")
    cos_raw = rm.neighbor_cosine(H["h_src"], H["h_dst"])            # (S, E)
    pe_dim = H["pe_src"].shape[-1]
    if pe_dim == 0:
        ax.hist(cos_raw.reshape(-1), bins=60, color="#94a3b8", alpha=0.7, label="LSTM only")
        ax.axvline(cos_raw.mean(), color="k", ls="--", lw=1.2)
        ax.set_title(f"{name} · no map-PE (mode={pe_mode})\nneighbor cos {cos_raw.mean():.3f}")
        ax.set_xlabel("cos(h_src, h_dst)"); ax.set_ylabel("count"); ax.legend(fontsize=8)
        print(f"  {name:8s}  no map-PE — train with --sheaf_map_pe lappe|learned to populate this")
        continue
    cos_pe = rm.neighbor_cosine(_augment(H["h_src"], H["pe_src"]),
                                _augment(H["h_dst"], H["pe_dst"]))
    ax.hist(cos_raw.reshape(-1), bins=60, color="#94a3b8", alpha=0.6,
            label=f"LSTM only  (mean {cos_raw.mean():.3f})")
    ax.hist(cos_pe.reshape(-1), bins=60, color="#2563eb", alpha=0.6,
            label=f"LSTM + {pe_mode} PE  (mean {cos_pe.mean():.3f})")
    ax.axvline(cos_raw.mean(), color="#475569", ls="--", lw=1.2)
    ax.axvline(cos_pe.mean(), color="#1d4ed8", ls="--", lw=1.2)
    tag = "equal-weight" if PE_EQUAL_WEIGHT else "as fed"
    ax.set_title(f"{name} · neighbor cosine + PE ({pe_mode}, {tag})")
    ax.set_xlabel("cos(endpoint inputs)"); ax.set_ylabel("count"); ax.legend(fontsize=8)
    pe_ratio = (np.linalg.norm(H["pe_src"], axis=-1).mean() /
                (np.linalg.norm(H["h_src"], axis=-1).mean() + 1e-12))
    summary[name]["neighbor_cos_pe"] = float(cos_pe.mean())
    print(f"  {name:8s}  {pe_mode:7s}  neighbor cos: LSTM {cos_raw.mean():.3f} -> +PE {cos_pe.mean():.3f}  "
          f"(\u0394 {cos_pe.mean() - cos_raw.mean():+.3f};  \u2016pe\u2016/\u2016h\u2016 \u2248 {pe_ratio:.2f})")
fig.suptitle("Neighbor cosine with the map positional encoding \u2014 do nearby nodes stay distinguishable to the MLP?",
             fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

### §6b · Positional encoding — how much diversity does it add?

When the restriction MLP is fed a LapPE (`restriction_node_pe_rank`), its input becomes
`[h_src ‖ pe_src ‖ h_dst ‖ pe_dst]`. Since the LSTM embeddings are near-collapsed, the no-PE
inputs pile up in a low-dimensional blob; the fixed, node-unique PE fans them apart, giving the
MLP room to emit distinct maps. Runs only for checkpoints trained with a restriction-map PE
(`map_pe` present). Two views per dataset: PCA scatter of the edge-input cloud **without** vs
**with** PE (coloured by source node — PE should separate the node clusters), and the set
effective rank + node-**separability** gain.

In [ ]:
# any_pe = False
# for name, rec in HARVEST.items():
#     gl = rec["model"].dynamics.graph_laplacian
#     map_pe = getattr(gl, "map_pe", None)
#     if map_pe is None:
#         print(f"{name}: no restriction-map PE (map_pe) — skipping.")
#         continue
#     any_pe = True
#     pe = np.asarray(map_pe.detach().cpu().numpy())                  # (N, r)
#     H = rec["H"]; src, dst = H["src"], H["dst"]; S = H["h_src"].shape[0]
#     pe_src = np.broadcast_to(pe[src], (S,) + pe[src].shape)         # (S, E, r)
#     pe_dst = np.broadcast_to(pe[dst], (S,) + pe[dst].shape)
#     div = rm.pe_input_diversity(H["h_src"], H["h_dst"], pe_src, pe_dst)
#     srclab = np.broadcast_to(src, (S, len(src))).reshape(-1)
#     sep_no = rm.class_separability(div["X_nope"], srclab)
#     sep_pe = rm.class_separability(div["X_pe"], srclab)

#     fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
#     ax[0].scatter(div["coords_nope"][:, 0], div["coords_nope"][:, 1], s=5, c=srclab, cmap="tab20", alpha=0.5)
#     ax[0].set_title(f"{name} · inputs WITHOUT PE\neff-rank {div['effrank_nope']:.2f}, node-sep {sep_no:.2f}")
#     ax[1].scatter(div["coords_pe"][:, 0], div["coords_pe"][:, 1], s=5, c=srclab, cmap="tab20", alpha=0.5)
#     ax[1].set_title(f"{name} · inputs WITH PE\neff-rank {div['effrank_pe']:.2f}, node-sep {sep_pe:.2f}")
#     for a in ax[:2]:
#         a.set_xlabel("PC1"); a.set_ylabel("PC2")
#     ax[2].bar(["no PE", "with PE"], [div["effrank_nope"], div["effrank_pe"]], color=["#94a3b8", "#2563eb"])
#     ax[2].set_ylabel("input set effective rank"); ax[2].set_title("diversity gain")
#     fig.suptitle(f"{name}: positional encoding enlarges the restriction-MLP input space", fontweight="bold")
#     fig.tight_layout(); plt.show()
#     summary[name]["pe_effrank_gain"] = float(div["effrank_pe"] - div["effrank_nope"])
#     summary[name]["pe_sep_gain"] = float(sep_pe - sep_no)
#     print(f"  {name}: eff-rank {div['effrank_nope']:.2f} -> {div['effrank_pe']:.2f}  |  "
#           f"node separability {sep_no:.2f} -> {sep_pe:.2f}")
# if not any_pe:
#     print("(No restriction-map PE in any configured checkpoint — train with "
#           "--restriction_node_pe_rank to populate this section.)")

## §7 · Qualitative curves of fit — best / worst / random (test set)

Real forecasts on the **test split**: each model runs one short-horizon, non-autoregressive
prediction per test window (`dt = cfg["dt"]`), and every `(window, node)` trace is ranked by
its horizon MSE. For each dataset we show three **2×3 panels** — the 6 best-fit, 6 worst-fit,
and 6 randomly-sampled traces — one curve per subplot: shaded **context** (observed), the
**ground-truth** horizon, and the model **prediction** (dashed). Values are in the model's
z-scored space. Config: `FORECAST_BATCHES` test batches, `CTX_SHOW` context steps drawn.

In [ ]:
FORECAST_BATCHES = int(os.environ.get("FORECAST_BATCHES", "6"))
CTX_SHOW = int(os.environ.get("CTX_SHOW", "15"))

FORECASTS = {}
for name, rec in HARVEST.items():
    loader, split = pick_split(rec["loaders"], prefer=("test", "val", "train"))
    fc = rm.harvest_forecasts(rec["model"], loader, rec["ei"],
                              norm_stats=rec["ckpt"].get("norm_stats"), dt=rec["dt"],
                              max_batches=FORECAST_BATCHES, device=device, ctx_show=CTX_SHOW)
    FORECASTS[name] = dict(fc=fc, split=split)
    S, N = fc["mse"].shape
    print(f"{name:5s}  split={split}  windows={S}  nodes={N}  "
          f"traces={S * N}  mean MSE={fc['mse'].mean():.4f}  dt={rec['dt']}")

In [ ]:
def plot_curve_panel(fc, flat_idx, title):
    """2×3 panel; one (window, node) forecast trace per subplot."""
    ctx, tru, prd, mse = fc["ctx"], fc["true"], fc["pred"], fc["mse"]
    N, Lc, Ly = ctx.shape[1], ctx.shape[2], tru.shape[2]
    tctx, th = np.arange(-Lc, 0), np.arange(0, Ly)
    fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
    for ax, k in zip(axes.ravel(), flat_idx):
        b, j = divmod(int(k), N)
        ax.axvspan(-Lc, 0, color="#dbeafe", alpha=0.35, zorder=0)
        ax.axvspan(0, Ly - 1, color="#fef3c7", alpha=0.28, zorder=0)
        ax.plot(tctx, ctx[b, j], color="#2563eb", lw=1.8, label="context")
        ax.plot(th, tru[b, j], color="#0f766e", lw=2.1, label="ground truth")
        ax.plot(th, prd[b, j], color="#dc2626", lw=1.8, ls="--", label="prediction")
        ax.axvline(0, color="#4b5563", ls=":", lw=1.0)
        ax.set_title(f"win {b} · node {j}   MSE={mse[b, j]:.3f}", fontsize=10)
        ax.grid(alpha=0.35)
    axes[0, 0].legend(fontsize=8, loc="upper left")
    for ax in axes[-1]:
        ax.set_xlabel("step (0 = forecast start)")
    fig.suptitle(title, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


rng_fit = np.random.default_rng(0)
for name, rc in FORECASTS.items():
    fc = rc["fc"]; N = fc["mse"].shape[1]
    order = np.argsort(fc["mse"].reshape(-1))          # ascending MSE over all (window, node)
    best = order[:6]
    worst = order[-6:][::-1]
    mid = order[6:-6]
    rand = (rng_fit.choice(mid, size=min(6, len(mid)), replace=False) if len(mid) else order[:6])
    for label, idx in [("best fits", best), ("worst fits", worst), ("random sample", rand)]:
        plot_curve_panel(fc, idx, f"{name} · {label} — test forecasts ({rc['split']})")

## §8 · EEG-only — restriction-map norms on the scalp

Topographic node-link maps for the EEG arm (nose up), each **edge coloured by its
restriction-map Frobenius norm** `½(‖ρ_src‖_F + ‖ρ_dst‖_F)`. Built on
`data.lemon_dataset.montage_positions`.
Two views: (1) the **average** over test windows, (2) a 2×3 array of **per-window
snapshots** (not time-averaged) showing how the map field flexes with the input.

`SCALE01=True` min-max stretches each map's colours to `[0,1]` for clarity (per-panel — so
snapshots maximise within-map contrast but are not cross-comparable). `SCALE01=False` uses
an absolute scale anchored at 0 (shared across the snapshots — magnitudes comparable, but a
tightly-clustered field may look washed out).

In [ ]:
# --- scalp projection (azimuthal-equidistant, nose up) -------------------
# MNE-style topomap flattening: radius grows with polar angle from the vertex,
# so low electrodes (…9/…10, e.g. PO9/PO10) fan out into their proper ring
# instead of collapsing onto the ring above them under a naive drop-z view.
def scalp_xy(pos3d):
    p = np.asarray(pos3d, float)
    q = p - np.array([0.0, 0.0, p[:, 2].min()])   # origin at cap bottom, +z = vertex
    r = np.linalg.norm(q, axis=1); r[r == 0] = 1.0
    theta = np.arccos(np.clip(q[:, 2] / r, -1.0, 1.0))  # 0 at vertex → outward
    phi = np.arctan2(q[:, 1], q[:, 0])                  # +y = nose (up), +x = right
    return np.c_[theta * np.cos(phi), theta * np.sin(phi)]

from matplotlib.collections import LineCollection
from pathlib import Path as _Path

EEG_NAME = next((k for k, v in DATASETS.items() if v["kind"] == "eeg"), None)
SCALE01  = os.environ.get("TOPO_SCALE01", "1") == "1"   # stretch colours to [0,1] for clarity

def draw_topo_edges(ax, ei_np, pos2d, edge_vals, title, scale01=True, cmap="viridis",
                    vmin=None, vmax=None):
    """Head outline + edges coloured by edge_vals (optionally min-max stretched to [0,1])."""
    r = np.abs(pos2d).max() * 1.15
    th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(r * np.cos(th), r * np.sin(th), color="k", lw=1)                             # head
    ax.plot([-0.09 * r, 0, 0.09 * r], [r * 0.99, r * 1.12, r * 0.99], color="k", lw=1)   # nose
    v = np.asarray(edge_vals, dtype=float)
    if scale01:
        lo, hi = np.nanmin(v), np.nanmax(v)
        v = (v - lo) / (hi - lo + 1e-12); vmin, vmax = 0.0, 1.0
    else:
        vmin = 0.0 if vmin is None else vmin
        vmax = np.nanmax(v) if vmax is None else vmax
    segs = [[(pos2d[s, 0], pos2d[s, 1]), (pos2d[d, 0], pos2d[d, 1])]
            for s, d in zip(ei_np[0], ei_np[1])]
    lc = LineCollection(segs, cmap=cmap, norm=plt.Normalize(vmin, vmax), lw=1.6, alpha=0.85)
    lc.set_array(v); ax.add_collection(lc)
    ax.scatter(pos2d[:, 0], pos2d[:, 1], s=18, c="k", zorder=3)
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.axis("off")
    return lc

if EEG_NAME is None:
    print("No EEG dataset configured — skipping scalp maps.")
else:
    from data.lemon_dataset import montage_positions
    rec = HARVEST[EEG_NAME]
    ei_np = rec["ei"].detach().cpu().numpy()
    train_ds = rec["loaders"]["train"].dataset
    channels = list(train_ds.channels)
    assert len(channels) == rec["N"], (len(channels), rec["N"])
    pj = _Path(train_ds._samples[0][0]).parent / "montage_positions.json"
    positions = montage_positions(channels, positions_json=str(pj) if pj.exists() else None)
    pos2d = scalp_xy(positions)

    # per-(window, edge) map magnitude = mean of src & dst Frobenius norm
    R_src, R_dst = rec["H"]["rho_src"], rec["H"]["rho_dst"]     # (S, E, H, d_e)
    edge_frob_win = 0.5 * (rm.frob(R_src).reshape(R_src.shape[:2]) +
                           rm.frob(R_dst).reshape(R_dst.shape[:2]))   # (S, E)
    edge_frob_avg = edge_frob_win.mean(0)                             # (E,)

    fig, ax = plt.subplots(figsize=(6, 6))
    lc = draw_topo_edges(ax, ei_np, pos2d, edge_frob_avg,
                         f"{EEG_NAME}: avg edge ‖ρ‖_F over windows  (SCALE01={SCALE01})",
                         scale01=SCALE01)
    fig.colorbar(lc, ax=ax, fraction=0.046,
                 label="min–max scaled" if SCALE01 else "mean ‖ρ‖_F")
    fig.tight_layout(); plt.show()

In [ ]:
if EEG_NAME is not None:
    S = edge_frob_win.shape[0]
    snap = np.unique(np.linspace(0, S - 1, 6).astype(int))       # 6 evenly-spaced windows
    gmax = float(edge_frob_win[snap].max())                      # shared abs scale (SCALE01=False)
    fig, axes = plt.subplots(2, 3, figsize=(15, 9.6))
    lc_last = None
    for ax, w in zip(axes.ravel(), snap):
        lc_last = draw_topo_edges(ax, ei_np, pos2d, edge_frob_win[w], f"window {w}",
                                  scale01=SCALE01, vmin=0.0, vmax=(None if SCALE01 else gmax))
    for ax in axes.ravel()[len(snap):]:
        ax.axis("off")
    if not SCALE01 and lc_last is not None:
        fig.colorbar(lc_last, ax=axes, fraction=0.02, label="mean ‖ρ‖_F (shared)")
    fig.suptitle(f"{EEG_NAME}: per-window edge ‖ρ‖_F snapshots  (SCALE01={SCALE01})",
                 fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

### §8b · Animated scalp video across the whole timeseries (download for slides)

Renders the per-window edge-‖ρ‖_F field as a movie over the full window timeline. Two
versions, matching the two static arrays above:
- **temporal** (`scale="global"`, fixed colour scale) — whole-brain magnitude changes over
  time are comparable frame-to-frame.
- **intrabrain** (`scale="per_frame"`, per-frame stretch) — always-max regional contrast
  within each frame.

Output goes to `ANALYSIS_OUT_DIR`. `.mp4` uses ffmpeg (auto-falls back to `.gif` if ffmpeg
is missing); a **GIF** drops straight onto a Google Slide (plays in present mode), an **MP4**
uploads to Drive then Insert ▸ Video ▸ Google Drive. Knobs: `ANIM_FMT`, `ANIM_FPS`,
`ANIM_STRIDE`, `ANIM_MAX_FRAMES`.

In [ ]:
if EEG_NAME is None:
    print("No EEG dataset — skipping animation.")
else:
    OUT_DIR     = os.environ.get("ANALYSIS_OUT_DIR", "outputs/restriction_maps")
    os.makedirs(OUT_DIR, exist_ok=True)
    ANIM_FMT    = os.environ.get("ANIM_FMT", "mp4")           # mp4 (ffmpeg) or gif (pillow)
    ANIM_FPS    = int(os.environ.get("ANIM_FPS", "6"))
    ANIM_STRIDE = int(os.environ.get("ANIM_STRIDE", "1"))     # subsample the timeline
    ANIM_MAX    = int(os.environ.get("ANIM_MAX_FRAMES", "300"))

    for scale, tag, desc in [("global", "temporal", "fixed scale — magnitude over time"),
                             ("per_frame", "intrabrain", "per-frame stretch — regional contrast")]:
        path = os.path.join(OUT_DIR, f"{EEG_NAME}_scalp_{tag}.{ANIM_FMT}")
        wrote = rm.save_scalp_animation(
            edge_frob_win, ei_np, pos2d, path, fps=ANIM_FPS, scale=scale,
            stride=ANIM_STRIDE, max_frames=ANIM_MAX, title=f"edge ||rho||_F [{tag}]")
        print(f"[{tag}] {desc}\n    -> {wrote}")
    print("\nGIF: drop straight into a Google Slide (plays in present mode). "
          "MP4: upload to Drive, then Insert > Video > Google Drive.")

## §8c · One diffusion step: message passing vs node-local update

Each sheaf round is the operator `H ← H − step·L_F H` (`model/sheaf.py` `SheafLaplacian.diffuse`).
Writing the Laplacian out per node,

$$(L_F h)_i = \Big(\sum_e \rho_{i\to e}^\top \rho_{i\to e}\Big)\, h_i \;-\; \sum_e \rho_{i\to e}^\top \rho_{j\to e}\, h_j$$

- **Diagonal / self block** `D h` (first term) is a *per-node* linear map applied to `h_i` — **no
  neighbor information anywhere in it**. It can be large all on its own.
- **Off-diagonal / neighbor block** `A h` (second term) is the **only channel that carries message
  passing**.

So a big `‖L_F h‖` and a visibly nonzero update can be *entirely* diagonal — no diffusion at all.
`rm.sheaf_step_decomposition` reproduces the model's exact `_restrict`/`_pullback`/`index_add`
aggregation and splits it: `L_F h = D h + A h` identically (by linearity of the pullback). We
report the **one-step difference** `Δh = −step·L_F h`, and the **ratio `‖A h‖/‖D h‖`** (neighbor vs
self) — projected on the EEG scalp. Analyzed at round 0 (the only round when `sheaf_layers=1`).

In [ ]:
# Decompose one diffusion step into self (diagonal) vs neighbor (off-diagonal) blocks,
# pooled over probe windows. See rm.sheaf_step_decomposition for the exact-as-implemented split.
STEP_DECOMP = {}
for name, rec in HARVEST.items():
    loader, _ = pick_split(rec["loaders"])
    STEP_DECOMP[name] = rm.sheaf_step_decomposition_over_loader(
        rec["model"], loader, rec["ei"], norm_stats=rec["ckpt"].get("norm_stats"),
        max_batches=MAX_BATCHES, device=device, round_idx=0)

for name, d in STEP_DECOMP.items():
    lap_n, self_n = d["lap_norm"].mean(), d["diag_norm"].mean()
    nbr_n, ratio = d["off_norm"].mean(), d["ratio"].mean()
    summary[name]["step_lap_norm"]  = float(lap_n)
    summary[name]["step_self_norm"] = float(self_n)
    summary[name]["step_nbr_norm"]  = float(nbr_n)
    summary[name]["nbr_over_self"]  = float(ratio)
    print(f"{name:8s}  ‖L_F h‖={lap_n:6.3f}   self ‖D h‖={self_n:6.3f}   nbr ‖A h‖={nbr_n:6.3f}   "
          f"‖A h‖/‖D h‖={ratio:5.3f}   (step={d['step']:.2f})")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
names = list(STEP_DECOMP); x = np.arange(len(names)); w = 0.38
selfv = [STEP_DECOMP[n]["diag_norm"].mean() for n in names]
nbrv  = [STEP_DECOMP[n]["off_norm"].mean()  for n in names]
ax[0].bar(x - w/2, selfv, w, label="self  ‖D h‖  (node-local)", color="#94a3b8")
ax[0].bar(x + w/2, nbrv,  w, label="neighbor  ‖A h‖  (message passing)", color="#0f766e")
ax[0].set_xticks(x); ax[0].set_xticklabels(names); ax[0].set_ylabel("mean per-node L2 norm")
ax[0].set_title("Self (diagonal) vs neighbor (off-diagonal) block"); ax[0].legend(fontsize=8)
for n in names:
    r = STEP_DECOMP[n]["ratio"].ravel()
    ax[1].hist(r, bins=60, alpha=0.5, density=True, label=f"{n}  (median {np.median(r):.2f})")
ax[1].axvline(1.0, color="k", ls=":", lw=1, label="neighbor = self")
ax[1].set_xlabel("‖A h‖ / ‖D h‖   (neighbor / self)"); ax[1].set_ylabel("density")
ax[1].set_title("Is the update genuine message passing?"); ax[1].legend(fontsize=7)
fig.tight_layout(); plt.show()

In [ ]:
# Project the one-step decomposition on the EEG brain graph (per-node scalars, nose up).
from matplotlib.collections import LineCollection
from pathlib import Path as _Path

EEG_NAME = next((k for k, v in DATASETS.items() if v["kind"] == "eeg"), None)

def draw_topo_nodes(ax, ei_np, pos2d, node_vals, title, cmap="magma", vmin=None, vmax=None):
    """Head outline + grey graph edges + nodes coloured by a per-node scalar."""
    r = np.abs(pos2d).max() * 1.15
    th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(r * np.cos(th), r * np.sin(th), color="k", lw=1)
    ax.plot([-0.09 * r, 0, 0.09 * r], [r * 0.99, r * 1.12, r * 0.99], color="k", lw=1)
    segs = [[(pos2d[s, 0], pos2d[s, 1]), (pos2d[d, 0], pos2d[d, 1])]
            for s, d in zip(ei_np[0], ei_np[1])]
    ax.add_collection(LineCollection(segs, colors="#d4d4d4", lw=0.5, alpha=0.6, zorder=1))
    v = np.asarray(node_vals, float)
    sc = ax.scatter(pos2d[:, 0], pos2d[:, 1], s=70, c=v, cmap=cmap,
                    vmin=(np.nanmin(v) if vmin is None else vmin),
                    vmax=(np.nanmax(v) if vmax is None else vmax),
                    edgecolors="k", linewidths=0.4, zorder=3)
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.axis("off")
    return sc

if EEG_NAME is None:
    print("No EEG dataset configured — skipping the scalp projection of the step decomposition.")
else:
    from data.lemon_dataset import montage_positions
    rec = HARVEST[EEG_NAME]; d = STEP_DECOMP[EEG_NAME]
    L = rec["model"].dynamics.sheaf_layers
    ei_np = rec["ei"].detach().cpu().numpy()
    train_ds = rec["loaders"]["train"].dataset
    channels = list(train_ds.channels)
    assert len(channels) == rec["N"], (len(channels), rec["N"])
    pj = _Path(train_ds._samples[0][0]).parent / "montage_positions.json"
    positions = montage_positions(channels, positions_json=str(pj) if pj.exists() else None)
    pos2d = scalp_xy(positions)

    upd   = d["update_norm"].mean(0)   # ‖Δh‖ = step·‖L_F h‖ — the difference one step delivers
    selfn = d["diag_norm"].mean(0)     # ‖D h‖  self / node-local
    nbrn  = d["off_norm"].mean(0)      # ‖A h‖  neighbor / message passing
    ratio = d["ratio"].mean(0)         # ‖A h‖ / ‖D h‖

    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    panels = [
        (upd,   "one-step update  ‖Δh‖ = step·‖L_F h‖",              "viridis"),
        (ratio, "‖A h‖ / ‖D h‖   (neighbor vs self)",               "coolwarm"),
        (selfn, "self / diagonal block  ‖D h‖  (node-local)",        "magma"),
        (nbrn,  "neighbor / off-diagonal block  ‖A h‖  (msg passing)", "magma"),
    ]
    for ax, (vals, ttl, cmap) in zip(axes.ravel(), panels):
        sc = draw_topo_nodes(ax, ei_np, pos2d, vals, f"{EEG_NAME}: {ttl}", cmap=cmap)
        fig.colorbar(sc, ax=ax, fraction=0.046)
    fig.suptitle(f"{EEG_NAME} · one diffusion step on the scalp  "
                 f"(round 0 of {L} sheaf round(s), step={d['step']:.2f})", fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()

    frac_local = float((d["ratio"] < 0.1).mean())
    print(f"{EEG_NAME}: median ‖A h‖/‖D h‖ = {np.median(d['ratio']):.3f}  |  "
          f"{100 * frac_local:.1f}% of (window,node) updates are <0.1 → essentially node-local")

### §8c continued — where the update comes from, and how much it varies

The §8c scalp shows the *average* diffusion operator. Three follow-ups:

- **Is `‖Δh‖ = 14` actually large?** Only relative to `‖h‖`. And is the self/neighbor balance a
  graph effect or just edge count? So: relative update `‖Δh‖/‖h‖` on the scalp, and
  **degree-normalized** self/neighbor blocks with `E` printed per graph.
- **Which edges deliver the update?** Each edge feeds *two* endpoints with *different* messages
  (`ρ_src ≠ ρ_dst`), so we **split every edge into a src-half and a dst-half** and colour each by
  the update it delivers. Then contrast that true contribution against `‖ρ‖` (what §8 colours by):
  the map norm ignores the coboundary `δ`, so a big-map edge whose endpoints already agree
  delivers almost nothing.
- **How much does the operator vary?** The maps are data-dependent, so every window instantiates a
  different operator — per-edge variability across windows, plus a few random single-window
  snapshots (the edge-contribution analogue of §8's map-norm snapshots).

In [ ]:
# Relative update ‖Δh‖/‖h‖ + degree-normalized self/neighbor blocks (removes the edge-count confound).
from matplotlib.collections import LineCollection
from pathlib import Path as _Path

EEG_NAME = next((k for k, v in DATASETS.items() if v["kind"] == "eeg"), None)

# STEP_DECOMP predates the h_norm addition if §8c was run on the older module — refresh if so.
if STEP_DECOMP and "h_norm" not in next(iter(STEP_DECOMP.values())):
    for _n, _rec in HARVEST.items():
        _ld, _ = pick_split(_rec["loaders"])
        STEP_DECOMP[_n] = rm.sheaf_step_decomposition_over_loader(
            _rec["model"], _ld, _rec["ei"], norm_stats=_rec["ckpt"].get("norm_stats"),
            max_batches=MAX_BATCHES, device=device, round_idx=0)

def draw_topo_nodes(ax, ei_np, pos2d, node_vals, title, cmap="magma", vmin=None, vmax=None):
    r = np.abs(pos2d).max() * 1.15
    th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(r * np.cos(th), r * np.sin(th), color="k", lw=1)
    ax.plot([-0.09 * r, 0, 0.09 * r], [r * 0.99, r * 1.12, r * 0.99], color="k", lw=1)
    segs = [[(pos2d[s, 0], pos2d[s, 1]), (pos2d[d, 0], pos2d[d, 1])]
            for s, d in zip(ei_np[0], ei_np[1])]
    ax.add_collection(LineCollection(segs, colors="#d4d4d4", lw=0.5, alpha=0.6, zorder=1))
    v = np.asarray(node_vals, float)
    sc = ax.scatter(pos2d[:, 0], pos2d[:, 1], s=70, c=v, cmap=cmap,
                    vmin=(np.nanmin(v) if vmin is None else vmin),
                    vmax=(np.nanmax(v) if vmax is None else vmax),
                    edgecolors="k", linewidths=0.4, zorder=3)
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.axis("off")
    return sc

def _montage_pos(rec):
    from data.lemon_dataset import montage_positions
    train_ds = rec["loaders"]["train"].dataset
    channels = list(train_ds.channels)
    pj = _Path(train_ds._samples[0][0]).parent / "montage_positions.json"
    positions = montage_positions(channels, positions_json=str(pj) if pj.exists() else None)
    return scalp_xy(positions)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
print("Degree-normalized blocks (per-incident-edge average) — removes the edge-count confound:")
names = list(STEP_DECOMP); x = np.arange(len(names)); w = 0.38
self_pe, nbr_pe = [], []
for name in names:
    rec = HARVEST[name]; ei = rec["ei"].detach().cpu().numpy()
    deg = np.bincount(np.concatenate([ei[0], ei[1]]), minlength=rec["N"]).astype(float)
    deg[deg == 0] = np.nan
    d = STEP_DECOMP[name]
    self_pe.append(np.nanmean(d["diag_norm"].mean(0) / deg))
    nbr_pe.append(np.nanmean(d["off_norm"].mean(0) / deg))
    print(f"  {name:10s} E={rec['ei'].shape[1]:5d}  mean degree={np.nanmean(deg):5.1f}  "
          f"self/edge={self_pe[-1]:.3f}  nbr/edge={nbr_pe[-1]:.3f}")
ax[0].bar(x - w / 2, self_pe, w, label="self ‖D h‖ / degree", color="#94a3b8")
ax[0].bar(x + w / 2, nbr_pe, w, label="neighbor ‖A h‖ / degree", color="#0f766e")
ax[0].set_xticks(x); ax[0].set_xticklabels(names); ax[0].set_ylabel("per-edge block magnitude")
ax[0].set_title("Degree-normalized self vs neighbor block"); ax[0].legend(fontsize=8)

if EEG_NAME is not None:
    rec = HARVEST[EEG_NAME]; d = STEP_DECOMP[EEG_NAME]
    ei_np = rec["ei"].detach().cpu().numpy(); pos2d = _montage_pos(rec)
    rel_node = (d["update_norm"] / (d["h_norm"] + 1e-12)).mean(0)          # (N,)
    sc = draw_topo_nodes(ax[1], ei_np, pos2d, rel_node,
                         f"{EEG_NAME}: ‖Δh‖ / ‖h‖ (relative update)", cmap="viridis")
    fig.colorbar(sc, ax=ax[1], fraction=0.046)
    rel_all = (d["update_norm"] / (d["h_norm"] + 1e-12))
    print(f"\n{EEG_NAME}: ‖Δh‖/‖h‖  median={np.median(rel_all):.2f}  "
          f"node-mean max={rel_node.max():.2f}  "
          f"({'>1 ⇒ the sheaf step changes the node more than it preserves' if rel_node.max() > 1 else 'update stays below the state magnitude'})")
else:
    ax[1].axis("off")
fig.tight_layout(); plt.show()

In [ ]:
# Split each edge in two: the update it delivers to its src end vs its dst end (different messages),
# coloured by the true contribution, averaged over windows.
def draw_topo_half_edges(ax, src, dst, pos2d, c_to_src, c_to_dst, title, cmap="magma", vmax=None):
    r = np.abs(pos2d).max() * 1.15
    th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(r * np.cos(th), r * np.sin(th), color="k", lw=1)
    ax.plot([-0.09 * r, 0, 0.09 * r], [r * 0.99, r * 1.12, r * 0.99], color="k", lw=1)
    segs, vals = [], []
    for e in range(len(src)):
        ps, pd = pos2d[src[e]], pos2d[dst[e]]; mid = 0.5 * (ps + pd)
        segs.append([tuple(ps), tuple(mid)]); vals.append(c_to_src[e])   # src half
        segs.append([tuple(pd), tuple(mid)]); vals.append(c_to_dst[e])   # dst half
    vals = np.asarray(vals, float)
    vmax = np.nanpercentile(vals, 99) if vmax is None else vmax
    lc = LineCollection(segs, cmap=cmap, norm=plt.Normalize(0, vmax), lw=1.8, alpha=0.85)
    lc.set_array(vals); ax.add_collection(lc)
    ax.scatter(pos2d[:, 0], pos2d[:, 1], s=14, c="k", zorder=3)
    ax.set_title(title, fontsize=10); ax.set_aspect("equal"); ax.axis("off")
    return lc

EDGE_CONTRIB = {}
if EEG_NAME is not None:
    rec = HARVEST[EEG_NAME]; loader, _ = pick_split(rec["loaders"])
    ecd = rm.harvest_edge_contributions_over_loader(
        rec["model"], loader, rec["ei"], norm_stats=rec["ckpt"].get("norm_stats"),
        max_batches=MAX_BATCHES, device=device, round_idx=0)
    EDGE_CONTRIB[EEG_NAME] = ecd
    pos2d = _montage_pos(rec)
    cs, cd = ecd["c_to_src"].mean(0), ecd["c_to_dst"].mean(0)             # (E,) avg over windows
    fig, ax = plt.subplots(figsize=(6.6, 6.6))
    lc = draw_topo_half_edges(ax, ecd["src"], ecd["dst"], pos2d, cs, cd,
                              f"{EEG_NAME}: per-edge update contribution\n(each edge split into src / dst half)")
    fig.colorbar(lc, ax=ax, fraction=0.046, label="‖ message this edge adds to that endpoint ‖")
    fig.tight_layout(); plt.show()
    print(f"{EEG_NAME}: {len(ecd['src'])} directed edges -> {2 * len(ecd['src'])} half-edge contributions | "
          f"mean ‖msg→src‖={cs.mean():.3f}  ‖msg→dst‖={cd.mean():.3f}")
else:
    print("No EEG dataset configured for the edge-contribution scalp.")

In [ ]:
# Why the contribution is a better metric than ‖ρ‖: the map norm ignores δ (the coboundary that
# actually flows). Each point is a half-edge, averaged over windows.
if EEG_NAME is not None:
    ecd = EDGE_CONTRIB[EEG_NAME]
    norm   = np.concatenate([ecd["rho_src_frob"].mean(0), ecd["rho_dst_frob"].mean(0)])
    contrib = np.concatenate([ecd["c_to_src"].mean(0),    ecd["c_to_dst"].mean(0)])
    dnorm  = np.concatenate([ecd["delta_norm"].mean(0),   ecd["delta_norm"].mean(0)])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
    sc0 = ax[0].scatter(norm, contrib, s=10, alpha=0.4, c=dnorm, cmap="plasma")
    r_norm = np.corrcoef(norm, contrib)[0, 1]
    ax[0].set_xlabel("map Frobenius norm ‖ρ‖  (what §8 colours by)")
    ax[0].set_ylabel("true update contribution ‖msg‖")
    ax[0].set_title(f"contribution vs ‖ρ‖   (Pearson r={r_norm:.2f})")
    fig.colorbar(sc0, ax=ax[0], fraction=0.046, label="‖δ‖ (coboundary)")
    proxy = norm * dnorm
    r_proxy = np.corrcoef(proxy, contrib)[0, 1]
    ax[1].scatter(proxy, contrib, s=10, alpha=0.4, color="#0f766e")
    ax[1].set_xlabel("‖ρ‖ · ‖δ‖"); ax[1].set_ylabel("true contribution ‖msg‖")
    ax[1].set_title(f"‖ρ‖·‖δ‖ vs contribution   (Pearson r={r_proxy:.2f})")
    fig.tight_layout(); plt.show()
    print(f"{EEG_NAME}: ‖ρ‖ alone explains r²={r_norm ** 2:.2f} of the contribution; "
          f"‖ρ‖·‖δ‖ explains r²={r_proxy ** 2:.2f}. The map norm misses the coboundary δ — an edge "
          "with a big map but agreeing endpoints (small δ) delivers almost nothing to the update.")

In [ ]:
# The maps are data-dependent, so each window instantiates a DIFFERENT diffusion operator.
# Per-edge variability across windows + the spread of the whole-operator magnitude.
if EEG_NAME is not None:
    ecd = EDGE_CONTRIB[EEG_NAME]
    C = np.concatenate([ecd["c_to_src"], ecd["c_to_dst"]], axis=1)        # (S, 2E) half-edge contribs
    cv = C.std(0) / (C.mean(0) + 1e-12)                                    # (2E,) across-window CV
    per_window_total = C.sum(1)                                           # (S,) total update magnitude
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    ax[0].hist(cv, bins=60, color="#6d28d9", alpha=0.85)
    ax[0].axvline(np.median(cv), color="crimson", ls="--", label=f"median CV {np.median(cv):.2f}")
    ax[0].set_xlabel("per-edge coefficient of variation across windows")
    ax[0].set_ylabel("count"); ax[0].set_title("How much each edge's contribution varies")
    ax[0].legend(fontsize=8)
    ax[1].hist(per_window_total / per_window_total.mean(), bins=40, color="#0f766e", alpha=0.85)
    ax[1].set_xlabel("per-window total contribution / mean"); ax[1].set_ylabel("count")
    ax[1].set_title("Spread of the whole diffusion operator across windows")
    fig.tight_layout(); plt.show()
    print(f"{EEG_NAME}: median per-edge CV={np.median(cv):.2f}  (0 = a static operator; the LSTM+MLP "
          f"make it data-dependent). Whole-operator magnitude varies "
          f"{100 * per_window_total.std() / per_window_total.mean():.0f}% window-to-window.")

In [ ]:
# A few randomly sampled windows of the half-edge contribution field — qualitative variation
# (mirrors §8's per-window map-norm snapshots, but coloured by contribution; shared colour scale).
if EEG_NAME is not None:
    ecd = EDGE_CONTRIB[EEG_NAME]; rec = HARVEST[EEG_NAME]; pos2d = _montage_pos(rec)
    S = ecd["c_to_src"].shape[0]
    rng = np.random.default_rng(0)
    snap = np.sort(rng.choice(S, size=min(6, S), replace=False))
    allc = np.concatenate([ecd["c_to_src"][snap], ecd["c_to_dst"][snap]])
    gmax = float(np.nanpercentile(allc, 99))
    fig, axes = plt.subplots(2, 3, figsize=(15, 9.6))
    for ax, wi in zip(axes.ravel(), snap):
        draw_topo_half_edges(ax, ecd["src"], ecd["dst"], pos2d,
                             ecd["c_to_src"][wi], ecd["c_to_dst"][wi], f"window {wi}", vmax=gmax)
    for ax in axes.ravel()[len(snap):]:
        ax.axis("off")
    fig.suptitle(f"{EEG_NAME}: per-edge update contribution — random window snapshots (shared scale)",
                 fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

## §8e · Effective directedness — does the diffusion recover the Granger arrow?

The Granger prior is **directed before** the LapPE symmetrization, and the sheaf diffuses on
that directed `edge_index` (LapPE is a separate node feature). So a directed edge `s → d`
(`score[s,d]` = *s Granger-causes d*, i.e. **d is the effect node**) delivers two *unequal*
updates each round — `c_to_dst` into `d`, `c_to_src` into `s` — because `ρ_src ≠ ρ_dst`. This
asks whether that imbalance lines up with the causal arrow.

- **Per-edge imbalance** `(to_dst − to_src)/(to_dst + to_src)` ∈ [−1, 1]: `> 0` ⇒ the update is
  pushed into the Granger-**effect** node — the operator's action agrees with the arrow.
- **Reciprocity control** (the caveat to watch for): a pair the Granger graph *already* carries
  in **both** directions has no preferred arrow, so its net node-flow should be ~symmetric; a
  **one-directional** pair should be lopsided. If a lopsided action showed up equally on
  bidirectional pairs, it would be a constant labelling artefact, not recovered directedness —
  so the uni-vs-bi contrast is the real test, not the raw imbalance.

Computed by `rm.diffusion_directedness` on the already-harvested `EDGE_CONTRIB` (no recompute;
`rec["ei"]` is the directed graph straight from the checkpoint).

In [ ]:
# Directedness readout from the harvested half-edge contributions (averaged over windows).
if EEG_NAME is not None:
    ecd = EDGE_CONTRIB[EEG_NAME]
    DD = rm.diffusion_directedness(ecd["src"], ecd["dst"], ecd["c_to_src"], ecd["c_to_dst"])
    s = DD["summary"]
    print(f"{EEG_NAME}: {s['n_edges']} directed edges over {s['n_pairs']} unordered pairs "
          f"({s['n_uni']} one-directional, {s['n_bi']} bidirectional; "
          f"{100 * s['frac_reciprocal_edges']:.0f}% of edges reciprocated)")
    print(f"  directional agreement on uni pairs (update flows into the Granger-EFFECT node): "
          f"{100 * s['agreement_rate_uni']:.0f}%   (50% = no directed recovery)")
    print(f"  pair-flow asymmetry median:  uni={s['median_asym_uni']:.2f}  bi={s['median_asym_bi']:.2f}"
          f"   (expect uni > bi if the operator honours direction)")

    uni, bi = ~DD["bidirectional"], DD["bidirectional"]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
    # (1) pair-flow asymmetry: one-directional vs bidirectional Granger pairs
    groups = [("one-directional", DD["pair_asym"][uni]), ("bidirectional", DD["pair_asym"][bi])]
    groups = [(lbl, v) for lbl, v in groups if len(v) > 0]
    if groups:
        ax[0].violinplot([v for _, v in groups], showmedians=True, showextrema=False)
        ax[0].set_xticks(range(1, len(groups) + 1))
        ax[0].set_xticklabels([f"{lbl}\n(n={len(v)})" for lbl, v in groups])
    ax[0].set_ylabel("pair-flow asymmetry  |Δ| / Σ"); ax[0].set_ylim(0, 1)
    ax[0].set_title("operator asymmetry vs Granger reciprocity")
    # (2) per-edge imbalance for one-directional edges; >0 = drives the effect node
    ue = ~DD["reciprocal"]
    vals = DD["edge_imbalance"][ue]
    ax[1].hist(vals, bins=30, color="#2563eb", alpha=0.85)
    ax[1].axvline(0, color="k", lw=1)
    ax[1].axvline(vals.mean(), color="crimson", ls="--", label=f"mean {vals.mean():+.2f}")
    ax[1].set_xlabel("edge imbalance  (to_dst − to_src)/(to_dst + to_src)\n"
                     "← drives cause        drives effect →")
    ax[1].set_ylabel("count (one-directional edges)")
    ax[1].set_title(f"where the update goes  ·  agree={100 * s['agreement_rate_uni']:.0f}%")
    ax[1].legend(fontsize=8)
    fig.tight_layout(); plt.show()
else:
    print("No EEG dataset configured — skipping the directedness analysis.")

In [ ]:
# Scalp view. Left: one-directional Granger edges as arrows pointing to the endpoint the
# operator drives harder (colour = |imbalance|); bidirectional pairs as grey undirected links.
# Right: per-node source(+)/sink(−) score  net = drives-neighbours − is-driven.
if EEG_NAME is not None:
    from matplotlib.cm import ScalarMappable
    rec = HARVEST[EEG_NAME]; pos2d = _montage_pos(rec)
    src, dst = ecd["src"], ecd["dst"]
    imb, recp = DD["edge_imbalance"], DD["reciprocal"]
    r = np.abs(pos2d).max() * 1.15; th = np.linspace(0, 2 * np.pi, 200)
    cmap = plt.get_cmap("viridis")
    vmax = float(np.nanpercentile(np.abs(imb[~recp]), 95)) if (~recp).any() else 1.0
    cnorm = plt.Normalize(0, max(vmax, 1e-6))

    fig, ax = plt.subplots(1, 2, figsize=(13.5, 6.8))
    for a in ax:
        a.plot(r * np.cos(th), r * np.sin(th), "k", lw=1)
        a.plot([-0.09 * r, 0, 0.09 * r], [r * 0.99, r * 1.12, r * 0.99], "k", lw=1)  # nose
        a.set_aspect("equal"); a.axis("off")

    seen = set()
    for e in range(len(src)):
        s_, d_ = int(src[e]), int(dst[e]); ps, pd = pos2d[s_], pos2d[d_]
        if recp[e]:
            key = (min(s_, d_), max(s_, d_))
            if key in seen:
                continue
            seen.add(key)
            ax[0].plot([ps[0], pd[0]], [ps[1], pd[1]], color="#cbd5e1", lw=1.2, zorder=1)
        else:
            p0, p1 = (ps, pd) if imb[e] >= 0 else (pd, ps)   # arrow -> endpoint driven harder
            ax[0].annotate("", xy=(p1[0], p1[1]), xytext=(p0[0], p0[1]),
                           arrowprops=dict(arrowstyle="-|>", lw=1.6,
                                           color=cmap(cnorm(abs(imb[e]))), alpha=0.9), zorder=2)
    ax[0].scatter(pos2d[:, 0], pos2d[:, 1], s=22, c="k", zorder=3)
    ax[0].set_title(f"{EEG_NAME}: directed operator action\n"
                    "arrow → endpoint driven harder  ·  grey = bidirectional Granger")
    fig.colorbar(ScalarMappable(norm=cnorm, cmap=cmap), ax=ax[0], fraction=0.046,
                 label="|edge imbalance|")

    net = DD["net_source"]; nlim = float(np.nanpercentile(np.abs(net), 98)) or 1.0
    for e in range(len(src)):
        ps, pd = pos2d[int(src[e])], pos2d[int(dst[e])]
        ax[1].plot([ps[0], pd[0]], [ps[1], pd[1]], color="#e5e7eb", lw=0.5, zorder=1)
    sc = ax[1].scatter(pos2d[:, 0], pos2d[:, 1], s=90, c=net, cmap="coolwarm",
                       vmin=-nlim, vmax=nlim, zorder=3, edgecolors="k", linewidths=0.4)
    ax[1].set_title("per-node source(+) / sink(−)\nnet = drives neighbours − is driven")
    fig.colorbar(sc, ax=ax[1], fraction=0.046, label="net_source")
    fig.tight_layout(); plt.show()

## §9 · Summary — fMRI vs EEG

One row per dataset. Read `map_spread ≈ in_spread ≪ xavier` + high `CKA` as *"maps are as
collapsed as the embeddings"*; `map_spread ≫ shuffled` as *"edge pairing carries real
information"*; `rot_frac` as the rotation/scaling balance; `neighbor_cos` / `sheaf_rel` from §6.

In [ ]:
cols = ["frob_mean", "frob_iso_mean", "effrank_mean", "rot_frac", "ortho_scale",
        "ortho_energy_frac", "ortho_smooth_ratio", "map_spread", "in_spread", "shuf_spread",
        "xav_spread", "cka", "jac_mean", "asym_mean", "neighbor_cos", "sheaf_rel",
        "nbr_over_self", "step_nbr_norm", "step_self_norm"]
try:
    import pandas as pd
    df = pd.DataFrame({name: {c: s.get(c, float("nan")) for c in cols}
                       for name, s in summary.items()}).T
    display(df.round(4))
except Exception:
    hdr = "dataset  " + "  ".join(f"{c:>12s}" for c in cols)
    print(hdr); print("-" * len(hdr))
    for name, s in summary.items():
        print(f"{name:8s} " + "  ".join(f"{s.get(c, float('nan')):>12.4f}" for c in cols))

## SVD-analysis time

In [ ]:
# === SVD-depth helpers (self-contained; numpy only) ==========================
import numpy as np

def svals_batch(R):
    """(..., H, d_e) -> singular values (P, k), P=prod(leading), k=min(H,d_e)."""
    R = np.asarray(R, dtype=np.float64); H, De = R.shape[-2], R.shape[-1]
    return np.linalg.svd(R.reshape(-1, H, De), compute_uv=False)

def _pr(lam):                                   # participation ratio on eigenvalues
    lam = np.asarray(lam, dtype=np.float64)
    return (lam.sum(-1) ** 2) / np.clip((lam ** 2).sum(-1), 1e-30, None)

def per_map_rank(R):    return _pr(svals_batch(R) ** 2)                    # (P,)  §3 question
def mean_map_spectrum(R):
    R = np.asarray(R, dtype=np.float64); H, De = R.shape[-2], R.shape[-1]
    return np.linalg.svd(R.reshape(-1, H, De).mean(0), compute_uv=False)
def mean_map_rank(R):   return float(_pr(mean_map_spectrum(R)[None] ** 2)[0])

def set_rank(R, center=True):                   # §5 question (map diversity / manifold dim)
    R = np.asarray(R, dtype=np.float64)
    Xf = R.reshape(-1, R.shape[-2] * R.shape[-1])
    if center: Xf = Xf - Xf.mean(0, keepdims=True)
    return float(_pr(np.linalg.svd(Xf, compute_uv=False) ** 2))

def random_maps(shape, n, kind="xavier", frob=None, rng=None):
    rng = np.random.default_rng() if rng is None else rng; H, De = shape
    if kind == "xavier":
        M = rng.normal(0.0, np.sqrt(2.0 / (H + De)), size=(n, H, De))
    else:                                       # orthogonal
        M = np.empty((n, H, De))
        for i in range(n):
            U, _, Vt = np.linalg.svd(rng.normal(size=(H, De)), full_matrices=False)
            M[i] = U @ Vt
    if frob is not None:
        M = M * (frob / np.clip(np.sqrt((M ** 2).sum((-2, -1), keepdims=True)), 1e-30, None))
    return M

def matched_mean_spectrum(shape, n, kind="xavier", frob=None, reps=20, seed=0):
    """Spectrum of n random maps averaged the SAME way — the correct control for a
    trained mean-map built by averaging n maps."""
    rng = np.random.default_rng(seed)
    S = np.stack([np.linalg.svd(random_maps(shape, n, kind, frob, rng).mean(0),
                                compute_uv=False) for _ in range(reps)])
    return S.mean(0), S.std(0)

def summarize(R, kind="xavier"):
    R = np.asarray(R, dtype=np.float64); H, De = R.shape[-2], R.shape[-1]
    Rf = R.reshape(-1, H, De); n = Rf.shape[0]
    frob = float(np.sqrt((Rf ** 2).sum((-2, -1)).mean()))
    Rr = random_maps((H, De), n, kind, frob=frob, rng=np.random.default_rng(0))
    return dict(per_map_rank=float(per_map_rank(R).mean()), mean_map_rank=mean_map_rank(R),
                set_rank=set_rank(R), rand_per_map=float(per_map_rank(Rr).mean()),
                rand_mean_map=mean_map_rank(Rr), rand_set=set_rank(Rr),
                full_rank=min(H, De), n_maps=n)

In [ ]:
# === singular-vector helpers (append to the SVD-depth helpers) ===============
def left_singular_vectors(R, k=None):
    """Top-k LEFT singular vectors of each map. Columns live in R^H = the node /
    LSTM hidden space (the map's DOMAIN), so these are the directions in LSTM
    space that the map reads from. Returns U (P, H, r), s (P, r)."""
    R = np.asarray(R, float); H, De = R.shape[-2], R.shape[-1]
    U, s, _ = np.linalg.svd(R.reshape(-1, H, De), full_matrices=False)
    r = s.shape[-1] if k is None else min(k, s.shape[-1])
    return U[..., :r], s[..., :r]

def sign_canon(V):                       # kill SVD sign ambiguity: max-|component| positive
    idx = np.argmax(np.abs(V), -1)
    sgn = np.sign(np.take_along_axis(V, idx[..., None], -1))[..., 0]
    return V * np.where(sgn == 0, 1.0, sgn)[..., None]

def pca_fit(X, k):
    X = np.asarray(X, float); mu = X.mean(0)
    _, s, Vt = np.linalg.svd(X - mu, full_matrices=False)
    return Vt[:k], mu, (s**2 / (s**2).sum())[:k]      # components (k,D), mean, evr

In [ ]:
def shared_dir(Uk):
    """Sign-robust consensus direction of unit vectors Uk (P, H): top eigenvector of
    E[u uᵀ]. Returns (w, concentration = top-eig fraction, mean|cos(u, w)|).
    concentration≈1 -> a single shared axis (any lobes in the u-scatter are a sign
    artifact); two large eigenvalues -> a genuine second direction."""
    S = (Uk.T @ Uk) / Uk.shape[0]
    ev, V = np.linalg.eigh(S)
    w = V[:, -1]; conc = ev[-1] / ev.sum(); mcos = np.abs(Uk @ w).mean()
    if w[np.argmax(np.abs(w))] < 0: w = -w        # display sign only (not meaningful)
    return w, float(conc), float(mcos)

In [ ]:
import matplotlib.pyplot as plt
STATS = {g: summarize(HARVEST[g]["H"]["rho_src"]) for g in HARVEST}   # reused below

fig, ax = plt.subplots(figsize=(1.6 + 3.0 * len(HARVEST), 4.2))
groups = list(HARVEST); x = np.arange(len(groups)); w = 0.12
for j, key in enumerate(["per_map_rank", "mean_map_rank", "set_rank"]):
    ax.bar(x + (j - 2.5) * w, [STATS[g][key] for g in groups], w, label=key.replace("_", " "))
for j, key in enumerate(["rand_per_map", "rand_mean_map", "rand_set"]):
    ax.bar(x + (j + 0.5) * w, [STATS[g][key] for g in groups], w, hatch="//", alpha=0.55,
           label=key.replace("rand_", "random ").replace("_", " "))
full = STATS[groups[0]]["full_rank"]
ax.axhline(full, color="k", ls=":", lw=1, label=f"full rank ({full})")
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylabel("effective rank (log)")
ax.set_title("per-map (filtering) · mean-map (shared) · set (diversity)")
ax.legend(fontsize=7.5, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
fig.tight_layout(); plt.show()

print(f"{'dataset':14s}{'per-map':>9s}{'mean-map':>9s}{'set':>8s}   "
      f"{'r per':>7s}{'r mean':>8s}{'r set':>8s}")
for g in groups:
    s = STATS[g]
    print(f"{g:14s}{s['per_map_rank']:9.2f}{s['mean_map_rank']:9.2f}{s['set_rank']:8.1f}   "
          f"{s['rand_per_map']:7.2f}{s['rand_mean_map']:8.2f}{s['rand_set']:8.1f}")
# per-map HIGH + set LOW  -> regime B: individually rich maps, low-diversity population.
# per-map LOW             -> genuine per-connection filtering.
# mean-map >> random mean -> shared structure survives averaging.

In [ ]:
n = len(HARVEST)
fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 4.0), squeeze=False)
for ax, (name, rec) in zip(axes[0], HARVEST.items()):
    R = np.asarray(rec["H"]["rho_src"], float); H, De = R.shape[-2], R.shape[-1]
    P = R.reshape(-1, H, De).shape[0]
    frob = float(np.sqrt((R.reshape(-1, H, De) ** 2).sum((-2, -1)).mean()))
    xs = np.arange(1, min(H, De) + 1)
    ax.semilogy(xs, np.clip(mean_map_spectrum(R), 1e-8, None), color="#1f77b4", lw=2.2,
                label="trained mean-map")
    m, sd = matched_mean_spectrum((H, De), P, "xavier", frob=frob, reps=20)
    ax.semilogy(xs, np.clip(m, 1e-8, None), "k--", lw=1.6, label=f"random, avg of {P}")
    ax.fill_between(xs, np.clip(m - sd, 1e-8, None), m + sd, color="k", alpha=0.15)
    one = random_maps((H, De), 1, "xavier", frob=frob, rng=np.random.default_rng(0))[0]
    ax.semilogy(xs, np.linalg.svd(one, compute_uv=False), ":", color="gray", lw=1.2,
                label="single random map (UNFAIR)")
    ax.set_title(f"{name} · mean-map spectrum"); ax.set_xlabel("singular index")
    ax.set_ylabel("σ (log)"); ax.legend(fontsize=7.5)
fig.tight_layout(); plt.show()
# trained ON the dashed floor -> mean is pure cancellation (averaging collapsed it).
# trained ABOVE the floor     -> shared structure genuinely survives = real signal.

In [ ]:
if EEG_NAME is not None:
    R = np.asarray(HARVEST[EEG_NAME]["H"]["rho_src"], float)      # (S, E, H, d_e)
    S, E = R.shape[0], R.shape[1]
    edge_rank = per_map_rank(R).reshape(S, E).mean(0)             # (E,) time-averaged
    edge_frob = np.sqrt((R ** 2).sum((-2, -1))).mean(0)          # (E,) gain, for contrast

    fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 6))
    draw_topo_edges(a0, ei_np, pos2d, edge_frob, f"{EEG_NAME}: edge ‖ρ‖_F  (gain)",
                    scale01=True, cmap="viridis")
    draw_topo_edges(a1, ei_np, pos2d, edge_rank,
                    f"{EEG_NAME}: edge effective rank  (channel capacity)",
                    scale01=True, cmap="magma")
    fig.suptitle("Same edge, two properties: coupling strength vs bandwidth")
    fig.tight_layout(); plt.show()
    print(f"per-edge rank: min={edge_rank.min():.2f}  max={edge_rank.max():.2f}  "
          f"mean={edge_rank.mean():.2f}  (full={min(R.shape[-2], R.shape[-1])})")


In [ ]:
fig, axes = plt.subplots(1, len(HARVEST), figsize=(5.6 * len(HARVEST), 4.2), squeeze=False)
for ax, (name, rec) in zip(axes[0], HARVEST.items()):
    R = np.asarray(rec["H"]["rho_src"], float); H, De = R.shape[-2], R.shape[-1]
    P = R.reshape(-1, H, De).shape[0]
    xav = np.random.default_rng(0).normal(0, np.sqrt(2 / (H + De)), size=(P, H, De))
    sL, sX = svals_batch(R), svals_batch(xav)
    idx = np.arange(sL.shape[1])
    ax.errorbar(idx,       sL.mean(0), yerr=sL.std(0), fmt='o', ms=2.5, lw=.8,
                color="#1f77b4", label="learned")
    ax.errorbar(idx + .25, sX.mean(0), yerr=sX.std(0), fmt='o', ms=2.5, lw=.8,
                color="#ff7f0e", alpha=.8, label="Xavier init")
    ax.set_title(f"{name}: σ per index"); ax.set_xlabel("σ index")
    ax.set_ylabel("σ value"); ax.legend()
fig.tight_layout(); plt.show()
# Xavier at its natural init scale (not norm-matched) reproduces the slide's
# "learned collapses / random stays broad". For a pure SHAPE comparison, divide
# each spectrum by its σ₀ before plotting.

In [ ]:
fig, axes = plt.subplots(1, len(HARVEST), figsize=(5.2 * len(HARVEST), 4.6), squeeze=False)
for ax, (name, rec) in zip(axes[0], HARVEST.items()):
    R = np.asarray(rec["H"]["rho_src"], float); H, De = R.shape[-2], R.shape[-1]
    P = R.reshape(-1, H, De).shape[0]
    xav = np.random.default_rng(0).normal(0, np.sqrt(2 / (H + De)), size=(P, H, De))
    u1L = sign_canon(left_singular_vectors(R,   1)[0][:, :, 0])     # (P, H)
    u1X = sign_canon(left_singular_vectors(xav, 1)[0][:, :, 0])
    comp, mu, evr = pca_fit(u1L, 2)                                 # fit on learned
    zL, zX = (u1L - mu) @ comp.T, (u1X - mu) @ comp.T               # same frame
    ax.scatter(zX[:, 0], zX[:, 1], s=6, c="#ff7f0e", alpha=.35, label="Xavier u₁")
    ax.scatter(zL[:, 0], zL[:, 1], s=6, c="#1f77b4", alpha=.55, label="learned u₁")
    ax.set_title(f"{name}: maps by u₁  (evr {evr[0]:.2f}, {evr[1]:.2f})")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()
# To see per-edge structure in the learned cloud, color it:
#   S, E = R.shape[0]... actually rec["H"]["rho_src"].shape[:2];  c=np.tile(np.arange(E), S)

In [ ]:
K_SING, K_PC = 6, 8
for name, rec in HARVEST.items():
    R = np.asarray(rec["H"]["rho_src"], float); H, De = R.shape[-2], R.shape[-1]
    hsrc = np.asarray(rec["H"]["h_src"], float).reshape(-1, H)       # LSTM cloud (P, H)
    U, s = left_singular_vectors(R, K_SING)                          # (P, H, K_SING)
    comp, mu, evr = pca_fit(hsrc, K_PC)                              # LSTM PCs (K_PC, H)

    align = np.abs(np.einsum("phk,jh->pkj", U, comp)).mean(0)        # (K_SING, K_PC) mean|cos|
    C = np.cov(hsrc, rowvar=False); trC = np.trace(C)
    cap_sing = np.einsum("phk,hg,pgk->pk", U, C, U).mean(0) / trC    # var captured per sing dir
    cap_pc, cap_rand = evr[:K_SING], 1.0 / H                         # ceiling / floor

    fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 4.4))
    im = a0.imshow(align, aspect="auto", cmap="magma", vmin=0, vmax=1)
    a0.set_xlabel("LSTM-space PC index"); a0.set_ylabel("map singular index")
    a0.set_title(f"{name}: |cos(map singular vec, LSTM PC)|")
    fig.colorbar(im, ax=a0, fraction=0.046)
    xi = np.arange(K_SING)
    a1.bar(xi - .2, cap_sing, .4, color="#1f77b4", label="map singular dir")
    a1.bar(xi + .2, cap_pc,   .4, color="#6ACC65", label="LSTM PC (ceiling)")
    a1.axhline(cap_rand, color="k", ls="--", lw=1.2, label="random dir (floor)")
    a1.set_xlabel("singular index"); a1.set_ylabel("frac of LSTM variance captured")
    a1.set_title("Dominant vs subordinate LSTM modes?"); a1.legend(fontsize=8)
    fig.tight_layout(); plt.show()

In [ ]:
DIRS = [0, 1, 2]                                   # singular indices -> "directions 1,2,3"
names = list(HARVEST)
fig, axes = plt.subplots(len(DIRS), len(names),
                         figsize=(6.2 * len(names), 2.6 * len(DIRS)), squeeze=False)
for cj, name in enumerate(names):
    R = np.asarray(HARVEST[name]["H"]["rho_src"], float)
    U, _ = left_singular_vectors(R, max(DIRS) + 1)          # (P, H, K)
    H = U.shape[1]
    for ri, k in enumerate(DIRS):
        w, conc, mcos = shared_dir(U[:, :, k])
        ax = axes[ri][cj]
        ax.bar(np.arange(H), w, color="#1f77b4")
        ax.axhline(0, color="k", lw=.6)
        ax.set_title(f"{name} · direction {k+1}   (conc={conc:.2f}, mean|cos|={mcos:.2f})",
                     fontsize=9)
        if ri == len(DIRS) - 1: ax.set_xlabel("LSTM hidden dim")
        ax.set_ylabel("component")
        peak = int(np.argmax(np.abs(w)))
        ax.annotate(f"peak dim {peak}", (peak, w[peak]), fontsize=7,
                    xytext=(3, 3), textcoords="offset points")
fig.suptitle("Shared left singular vector (sign-robust) — what LSTM dims each routing axis reads",
             fontsize=12, weight="bold")
fig.tight_layout(); plt.show()

# polarity verdict per direction/dataset
print(f"{'dataset':12s}{'dir':>4s}{'concentration':>14s}{'mean|cos|':>11s}   verdict")
for name in names:
    U, _ = left_singular_vectors(np.asarray(HARVEST[name]['H']['rho_src'], float), max(DIRS)+1)
    for k in DIRS:
        _, conc, mcos = shared_dir(U[:, :, k])
        v = "single shared axis (lobes = sign artifact)" if conc > 0.7 else \
            "spread / possible 2nd direction — check anatomy" if conc > 0.4 else "diffuse"
        print(f"{name:12s}{k+1:>4d}{conc:>14.2f}{mcos:>11.2f}   {v}")

## §10 · Net attention spectrum — singular-value-weighted left vectors, colored by biology

The `maps by u₁` plot only used the top left singular vector. The **net attention operator**
`P_L = U S Uᵀ` weights *every* left vector by its singular value, so it captures the whole
spectrum the map emphasizes (and is sign-invariant, so it can be PCA'd across edges). We
average it per edge over windows, PCA to 2D, and color by the source electrode's hemisphere
and lobe — does biology organize what the maps attend to?

In [ ]:
def _pca2(X):
    Xc = X - X.mean(0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T, (S[:2] ** 2 / (S ** 2).sum())

for name, rec in HARVEST.items():
    R = rec['H']['rho_src']; S, E, Hd, De = R.shape
    Pl = rm.net_attention_operator(R.reshape(-1, Hd, De)).reshape(S, E, Hd, Hd).mean(0)  # (E,Hd,Hd)
    pts, evr = _pca2(Pl.reshape(E, -1))
    ei_np = rec['ei'].detach().cpu().numpy()
    try:
        grp = rm.electrode_groups(list(rec['loaders']['train'].dataset.channels))
        hemi, lobe = grp['hemisphere'][ei_np[0]], grp['lobe'][ei_np[0]]
    except Exception as e:
        hemi = lobe = None; print(f'{name}: no channel labels ({e})')
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    for ax, lab, ttl in [(axes[0], hemi, 'src hemisphere'), (axes[1], lobe, 'src lobe')]:
        if lab is None:
            ax.scatter(pts[:, 0], pts[:, 1], s=10, alpha=0.6)
        else:
            for gname in np.unique(lab):
                mk = lab == gname
                ax.scatter(pts[mk, 0], pts[mk, 1], s=16, alpha=0.75, label=str(gname))
            ax.legend(fontsize=7, title=ttl)
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    fig.suptitle(f'{name}: net-attention operator PCA  (EVR {evr[0]:.2f}, {evr[1]:.2f})')
    fig.tight_layout(); plt.show()

## §11 · How much of an orthogonal matrix is a map, really?

A polar `rotation_frac` (e.g. 0.65) is a *ratio of deviations from the identity*, not an amount
of orthogonality. The singular-value split answers the actual question: a map is
`orthogonal_scale = mean(s)` × an orthogonal matrix, and that scaled-orthogonal approximation
explains `isometric_energy_frac = mean(s)²/mean(s²)` of its energy (the rest is anisotropic
stretch). So a map with mean isometry-norm ~0.72 is ~*0.72 of an orthogonal matrix* — not 0.2.

In [ ]:
fig, axes = plt.subplots(1, len(HARVEST), figsize=(5 * len(HARVEST), 4), squeeze=False)
for ax, (name, rec) in zip(axes[0], HARVEST.items()):
    oc = rm.orthogonal_content(rec['H']['rho_src'])
    ax.hist(oc['orthogonal_scale'], bins=50, alpha=0.75, color='#4878CF')
    ax.axvline(1.0, color='k', ls=':', label='orthogonal = 1')
    ax.axvline(oc['orthogonal_scale'].mean(), color='crimson', ls='--', label=f"mean {oc['orthogonal_scale'].mean():.2f}")
    ax.set_title(f"{name}: iso-frac {oc['isometric_energy_frac'].mean():.2f}", fontsize=9)
    ax.set_xlabel('orthogonal scale = mean singular value'); ax.legend(fontsize=7)
fig.suptitle('Orthogonal scale of each map  (mean s; 1.0 = a full orthogonal matrix)')
fig.tight_layout(); plt.show()

for name, rec in HARVEST.items():
    oc = rm.orthogonal_content(rec['H']['rho_src'])
    print(f"{name:6s} orthogonal_scale={oc['orthogonal_scale'].mean():.3f}  "
          f"isometric_energy_frac={oc['isometric_energy_frac'].mean():.3f}  "
          f"anisotropy={oc['anisotropy'].mean():.3f}")

### §11b · Is the orthogonal map field smoother than random?

Dirichlet energy of the orthogonal (polar) component over the graph: mean distance between the
maps of edges that share a node, vs the same maps randomly re-assigned to edges. `ratio < 1`
means adjacent edges carry *more similar* maps than chance — the model found a smoother, more
globally coherent geometry rather than independent per-edge rotations.

In [ ]:
print('orthogonal map-field smoothness (lower ratio = smoother than random):')
for name, rec in HARVEST.items():
    R = rec['H']['rho_src'].mean(0)                 # (E, Hd, De) window-mean
    Q = rm.polar_orthogonal(R)
    ei_np = rec['ei'].detach().cpu().numpy()
    pairs = np.stack([ei_np[0], ei_np[1]], 1)
    sm = rm.map_field_smoothness(Q, pairs, n_null=10)
    print(f"  {name:6s} learned={sm['learned']:.3f}  null={sm['null']:.3f}  "
          f"ratio={sm['smoothness_ratio']:.3f}  (adjacent-edge pairs {sm['n_adj']})")

## §12 · Do two conditions recover the same maps? (cross-condition Gram)

Load two runs into `DATASETS` (e.g. EC and EO). On their **shared** edges we take the
orthogonal component of each condition's per-edge map and form the cosine Gram. If the diagonal
(same edge across conditions) sits well above the off-diagonal (different edges), the two
conditions are recovering the *same edge-specific geometry* — a stronger claim than matching
loss. `separation ≈ 0` means the maps carry no condition-stable edge identity.

In [ ]:
items = list(HARVEST.items())
if len(items) < 2:
    print('load two conditions into DATASETS (e.g. EEG_ec and EEG_eo) for this cell.')
else:
    (n1, r1), (n2, r2) = items[0], items[1]
    R1, R2 = r1['H']['rho_src'].mean(0), r2['H']['rho_src'].mean(0)
    p1 = np.stack([r1['ei'][0].cpu().numpy(), r1['ei'][1].cpu().numpy()], 1)
    p2 = np.stack([r2['ei'][0].cpu().numpy(), r2['ei'][1].cpu().numpy()], 1)
    Ra, Rb, common = rm.align_edge_maps(p1, R1, p2, R2)
    if len(common) < 3:
        print(f'only {len(common)} shared edges between {n1} and {n2}.')
    else:
        cg = rm.cross_condition_gram(rm.polar_orthogonal(Ra), rm.polar_orthogonal(Rb))
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
        im = axes[0].imshow(cg['gram'], cmap='RdBu_r', vmin=-1, vmax=1)
        axes[0].set_title(f'cosine Gram: {n1} vs {n2}\n(diagonal = same edge)', fontsize=9)
        fig.colorbar(im, ax=axes[0], fraction=0.046)
        E = len(common)
        axes[1].hist(cg['gram'][~np.eye(E, dtype=bool)], bins=40, density=True, alpha=0.6, color='grey', label='different edges')
        axes[1].hist(cg['diag'], bins=20, density=True, alpha=0.75, color='#4878CF', label='same edge')
        axes[1].axvline(cg['diag_mean'], color='#4878CF', ls='--'); axes[1].axvline(cg['offdiag_mean'], color='grey', ls='--')
        axes[1].set_title(f"separation = {cg['separation']:+.3f}", fontsize=10); axes[1].set_xlabel('cosine'); axes[1].legend(fontsize=8)
        fig.suptitle(f'Same edge maps across conditions?  ({E} shared edges)')
        fig.tight_layout(); plt.show()

## §13 · Positional encoding: map diversity & discriminability (A vs B)

What the PE buys the model, made visual. Each color is one edge's maps across windows. Without
PE (cell A) the maps are slaved to the near-collapsed embeddings, so edges overlap and the
model cannot tell them apart; the PE (cell B) pins edge identity, so the per-edge clouds pull
into separated clusters. The title reports the between-vs-within separation ratio and a 1-NN
edge-identity accuracy (chance = 1/E) — both should jump with the PE.

In [ ]:
# def _pca2b(X):
#     Xc = X - X.mean(0); U, S, Vt = np.linalg.svd(Xc, full_matrices=False); return Xc @ Vt[:2].T

# fig, axes = plt.subplots(1, len(HARVEST), figsize=(5.2 * len(HARVEST), 4.7), squeeze=False)
# for ax, (name, rec) in zip(axes[0], HARVEST.items()):
#     R = rec['H']['rho_src']; S, E = R.shape[0], R.shape[1]
#     ecs = rm.edge_cluster_separation(R, subsample=2000)
#     ne = min(8, E); sel = np.linspace(0, E - 1, ne).astype(int)
#     sub = R[:, sel]                                  # (S, ne, Hd, De)
#     pts = _pca2b(rm.flatten_maps(sub.reshape(-1, R.shape[-2], R.shape[-1])))
#     lab = np.tile(sel, S)                            # row-major: index = s*ne + e_local
#     for e in sel:
#         mk = lab == e; ax.scatter(pts[mk, 0], pts[mk, 1], s=14, alpha=0.6)
#     ax.set_title(f"{name}\nsep-ratio {ecs['separation_ratio']:.1f}  "
#                  f"1-NN edge acc {ecs['nn_edge_acc']:.2f} (chance {ecs['chance']:.3f})", fontsize=9)
#     ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
# fig.suptitle('PE diversity & discriminability: each color = one edge; tight separated clusters = maps carry edge identity')
# fig.tight_layout(); plt.show()

## §14 · Net-operator routing axes — which LSTM dims the net attention reads

The §36 plot took the sign-robust consensus of *one* singular index. Here we do the same idea
on the **net attention operator** `E[P_L] = E[U S Uᵀ]`, whose eigenvectors are the
singular-value-weighted consensus routing axes (the whole spectrum at once, sign-free). Each
bar plot is one net axis over the LSTM hidden dims; the title gives its eigenvalue share.

In [ ]:
K_AX = 3
for name, rec in HARVEST.items():
    R = np.asarray(rec['H']['rho_src'], float); S, E, Hd, De = R.shape
    Pl_edge = rm.net_attention_operator(R.reshape(-1, Hd, De)).reshape(S, E, Hd, Hd).mean(0)  # (E,Hd,Hd)
    vecs, frac = rm.principal_axes(Pl_edge.mean(0), k=K_AX)          # global mean net operator
    fig, axes = plt.subplots(1, K_AX, figsize=(4.0 * K_AX, 3.2), squeeze=False)
    for a, i in zip(axes[0], range(K_AX)):
        a.bar(np.arange(Hd), vecs[:, i], color='#1f77b4'); a.axhline(0, color='k', lw=.6)
        peak = int(np.argmax(np.abs(vecs[:, i])))
        a.set_title(f'net axis {i+1}  ({frac[i]*100:.0f}% of net energy, peak dim {peak})', fontsize=9)
        a.set_xlabel('LSTM hidden dim'); a.set_ylabel('component')
    fig.suptitle(f'{name}: net-attention routing axes (E[U S Uᵀ] eigenvectors)', weight='bold')
    fig.tight_layout(); plt.show()

### §14b · Per-region routing axes

Break the net operator down by the source electrode's lobe: average `P_L` over the edges
leaving each region and take its leading axis. If regions route through *different* hidden dims,
the leading bars will differ region to region — evidence the maps carry region-specific geometry.

In [ ]:
for name, rec in HARVEST.items():
    R = np.asarray(rec['H']['rho_src'], float); S, E, Hd, De = R.shape
    Pl_edge = rm.net_attention_operator(R.reshape(-1, Hd, De)).reshape(S, E, Hd, Hd).mean(0)
    ei_np = rec['ei'].detach().cpu().numpy()
    try:
        lobe = rm.electrode_groups(list(rec['loaders']['train'].dataset.channels))['lobe'][ei_np[0]]
    except Exception as e:
        print(f'{name}: no channel labels ({e})'); continue
    regions = [g for g in ['frontal','central','parietal','temporal','occipital'] if (lobe==g).sum() >= 3]
    fig, axes = plt.subplots(1, len(regions), figsize=(3.4*len(regions), 3.0), squeeze=False)
    for a, g in zip(axes[0], regions):
        vecs, frac = rm.principal_axes(Pl_edge[lobe==g].mean(0), k=1)
        a.bar(np.arange(Hd), vecs[:, 0], color='#6ACC65'); a.axhline(0, color='k', lw=.6)
        a.set_title(f'{g}  (n={int((lobe==g).sum())}, {frac[0]*100:.0f}%)', fontsize=9)
        a.set_xlabel('hidden dim')
    fig.suptitle(f'{name}: leading net-attention axis per source lobe', weight='bold')
    fig.tight_layout(); plt.show()

### §14c · What each restriction map does to an LSTM vector — stretch + rotation

Any matrix acts as **rotation × stretch** (SVD `ρ = U S Vᵀ`). The earlier version of this cell
plotted `diag(E[P_L])` — it averaged the operator *across maps first*, and since each map is
low-rank onto a *different* set of directions, the average washed out to look uniform (and every
bar fell below Xavier only because trained maps have a smaller norm — a scale artefact, not
attenuation). Here we read the **per-map** SVD instead:

- **Left — stretching.** Each map's singular values, sorted then averaged, normalised to `σ₁`
  (scale-free), with an IQR error band. A steep decay = the map concentrates the LSTM vector onto a
  few directions. Compare the shape to Xavier-random maps, and to the averaged operator `E[P_L]`
  (dotted): that flat dotted line is exactly the artefact the old plot showed. Title = effective
  rank (learned vs Xavier).
- **Right — rotation vs stretch.** Polar content per map (mean ± std): *anisotropic stretch*
  (`orthogonal_content.anisotropy`; 0 = pure isotropic scaling / isometry, 1 = all energy in an
  uneven stretch) and *rotation fraction* (`svd_polar.rotation_frac`). The subtitle reports the
  overall scale `mean σ`, which is where the maps genuinely shrink.

In [ ]:
# Per-MAP SVD (sorted-then-averaged), NOT the averaged operator: the maps are each low-rank but
# onto DIFFERENT directions, so averaging U S Uᵀ first (old §14c) isotropised them. Singular values
# are normalised to σ1 so this compares SHAPE (anisotropy), not overall norm, against Xavier.
n = len(HARVEST)
fig, axes = plt.subplots(n, 2, figsize=(13, 4.3 * n), squeeze=False)
rng0 = np.random.default_rng(0)
for row, (name, rec) in enumerate(HARVEST.items()):
    R = np.asarray(rec["H"]["rho_src"], float)
    H, De = R.shape[-2], R.shape[-1]
    R2 = R.reshape(-1, H, De); P = R2.shape[0]
    xav = rng0.normal(0, np.sqrt(2 / (H + De)), size=(P, H, De))

    # ---- STRETCH: per-map singular spectrum, normalised to sigma_1 ----
    sv = rm.svdvals(R2); svn = sv / (sv[:, :1] + 1e-12)
    xsv = rm.svdvals(xav); xsvn = xsv / (xsv[:, :1] + 1e-12)
    er, xer = rm.effective_rank(R2), rm.effective_rank(xav)
    r = sv.shape[1]; k = np.arange(1, r + 1)
    m = svn.mean(0); lo, hi = np.percentile(svn, [25, 75], axis=0)
    M = rm.net_attention_operator(R2).mean(0)                       # averaged operator (old §14c)
    ev = np.linalg.eigvalsh(0.5 * (M + M.T))[::-1]; evn = ev / (ev[0] + 1e-12)

    a0 = axes[row, 0]
    a0.plot(k, m, "-o", ms=3, color="#1f77b4",
            label=f"learned per-map  (eff.rank {er.mean():.1f}\u00b1{er.std():.1f})")
    a0.fill_between(k, lo, hi, color="#1f77b4", alpha=0.2, label="learned IQR (25\u201375%)")
    a0.plot(k, xsvn.mean(0), "--", color="#888", label=f"Xavier per-map  (eff.rank {xer.mean():.1f})")
    a0.plot(k, evn[:r], ":", color="#d62728", lw=1.8, label="averaged operator E[P_L] (old \u00a714c \u2014 flat)")
    a0.set_xlabel("singular index k"); a0.set_ylabel(r"$\sigma_k/\sigma_1$  (stretch, scale-free)")
    a0.set_title(f"{name}: stretching spectrum"); a0.legend(fontsize=7)

    # ---- ROTATION vs anisotropic STRETCH: polar content, mean +/- std ----
    oc, spd = rm.orthogonal_content(R2), rm.svd_polar(R2)
    ocx, spdx = rm.orthogonal_content(xav), rm.svd_polar(xav)
    labels = ["anisotropic\nstretch", "rotation\nfrac"]
    L = [oc["anisotropy"].mean(), spd["rotation_frac"].mean()]
    Lerr = [oc["anisotropy"].std(), spd["rotation_frac"].std()]
    X = [ocx["anisotropy"].mean(), spdx["rotation_frac"].mean()]
    Xerr = [ocx["anisotropy"].std(), spdx["rotation_frac"].std()]
    a1 = axes[row, 1]; xx = np.arange(2); w = 0.36
    a1.bar(xx - w / 2, L, w, yerr=Lerr, capsize=4, color="#1f77b4", label="learned")
    a1.bar(xx + w / 2, X, w, yerr=Xerr, capsize=4, color="#bbb", label="Xavier")
    a1.set_xticks(xx); a1.set_xticklabels(labels); a1.set_ylabel("fraction"); a1.set_ylim(0, 1)
    a1.set_title(f"{name}: rotation vs stretch  (mean \u00b1 std)\n"
                 f"scale (mean \u03c3) learned {oc['orthogonal_scale'].mean():.2f} "
                 f"vs Xavier {ocx['orthogonal_scale'].mean():.2f}")
    a1.legend(fontsize=8)
fig.suptitle("\u00a714c \u00b7 What each restriction map does to an LSTM vector \u2014 SVD stretch + polar rotation",
             fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()

## §15 · What are the two groups? (norm / u₁ bimodality)

The map-norm histogram and the u₁ PCA both look bimodal. We split the per-edge norm at its
Otsu antimode and ask what separates the two modes — hemisphere, lobe, orthogonal scale, and
(the earlier u₁ PCA) — to name the two populations rather than just see them.

In [ ]:
import pandas as pd
for name, rec in HARVEST.items():
    R = np.asarray(rec['H']['rho_src'], float); S, E = R.shape[0], R.shape[1]
    edge_norm = rm.frob(R, 'isometry').reshape(S, E).mean(0)      # (E,) per-edge isometry norm
    thr = rm.bimodal_threshold(edge_norm)
    hi = edge_norm > thr
    oc = rm.orthogonal_content(R.mean(0))                          # per-edge orthogonal scale
    ei_np = rec['ei'].detach().cpu().numpy()
    try:
        grp = rm.electrode_groups(list(rec['loaders']['train'].dataset.channels))
        hemi, lobe = grp['hemisphere'][ei_np[0]], grp['lobe'][ei_np[0]]
    except Exception:
        hemi = lobe = np.array(['?'] * E)
    # histogram with the split
    fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 4.0))
    a0.hist(edge_norm[~hi], bins=30, alpha=0.7, color='#B47CC7', label=f'low  (n={int((~hi).sum())})')
    a0.hist(edge_norm[hi],  bins=30, alpha=0.7, color='#4878CF', label=f'high (n={int(hi.sum())})')
    a0.axvline(thr, color='k', ls='--', label=f'Otsu {thr:.2f}'); a0.set_xlabel('edge ‖ρ‖ [isometry]'); a0.legend(fontsize=8)
    a0.set_title(f'{name}: norm modes')
    # lobe composition of each mode
    lobes = [g for g in ['frontal','central','parietal','temporal','occipital','other'] if (lobe==g).any()]
    lo_frac = [ (lobe[~hi]==g).mean() for g in lobes ]; hi_frac = [ (lobe[hi]==g).mean() for g in lobes ]
    xp = np.arange(len(lobes)); a1.bar(xp-.2, lo_frac, .4, color='#B47CC7', label='low'); a1.bar(xp+.2, hi_frac, .4, color='#4878CF', label='high')
    a1.set_xticks(xp); a1.set_xticklabels(lobes, rotation=20); a1.set_ylabel('fraction of mode'); a1.set_title('source lobe by mode'); a1.legend(fontsize=8)
    fig.tight_layout(); plt.show()
    # summary table
    df = pd.DataFrame({
        'mode': ['low', 'high'],
        'n': [int((~hi).sum()), int(hi.sum())],
        'mean_norm': [edge_norm[~hi].mean(), edge_norm[hi].mean()],
        'orth_scale': [oc['orthogonal_scale'][~hi].mean(), oc['orthogonal_scale'][hi].mean()],
        'frac_L': [(hemi[~hi]=='L').mean(), (hemi[hi]=='L').mean()],
        'frac_R': [(hemi[~hi]=='R').mean(), (hemi[hi]=='R').mean()],
    }).set_index('mode').round(3)
    print(name); display(df)

## §16 · Raw-norm scalp & prune candidates

The §8 scalp min-max stretches each frame; here the edges are colored by **raw** isometry-norm
on a shared scale, so absolute magnitude is comparable and the weak (low-norm) edges — the ones
you'd trim when refining a prior graph — are directly eyeball-able. Below it: the sorted-norm
curve with the Otsu cut, and norm vs the Granger prior / electrode distance so you can see
*what kind* of connections the low-norm tail is.

In [ ]:
for name, rec in HARVEST.items():
    R = np.asarray(rec['H']['rho_src'], float); S, E = R.shape[0], R.shape[1]
    edge_norm = rm.frob(R, 'isometry').reshape(S, E).mean(0)
    thr = rm.bimodal_threshold(edge_norm)
    ei_np = rec['ei'].detach().cpu().numpy()
    try:
        channels = list(rec['loaders']['train'].dataset.channels)
        from data.lemon_dataset import montage_positions
        from pathlib import Path as _P
        pj = _P(rec['loaders']['train'].dataset._samples[0][0]).parent / 'montage_positions.json'
        pos2d_ = scalp_xy(montage_positions(channels, positions_json=str(pj) if pj.exists() else None))
    except Exception as e:
        print(f'{name}: no montage ({e})'); pos2d_ = None
    gs = rec['ckpt'].get('graph_score')
    gvals = np.asarray(gs)[ei_np[0], ei_np[1]] if gs is not None else None
    fig = plt.figure(figsize=(15, 4.2))
    if pos2d_ is not None:
        ax0 = fig.add_subplot(1, 3, 1)
        lc = draw_topo_edges(ax0, ei_np, pos2d_, edge_norm, f'{name}: raw ‖ρ‖ (shared scale)', scale01=False, cmap='viridis')
        fig.colorbar(lc, ax=ax0, fraction=0.046, label='‖ρ‖ [isometry]')
    ax1 = fig.add_subplot(1, 3, 2)
    order = np.argsort(edge_norm)
    ax1.plot(np.arange(E), edge_norm[order], lw=1.5); ax1.axhline(thr, color='crimson', ls='--', label=f'Otsu {thr:.2f}')
    ax1.axhline(1.0, color='k', ls=':', label='isometry 1'); ax1.set_xlabel('edge rank'); ax1.set_ylabel('‖ρ‖'); ax1.legend(fontsize=8)
    ax1.set_title(f'sorted norms — {int((edge_norm<thr).sum())}/{E} below cut')
    ax2 = fig.add_subplot(1, 3, 3)
    if gvals is not None:
        from scipy.stats import spearmanr
        rho, _ = spearmanr(edge_norm, gvals)
        ax2.scatter(gvals, edge_norm, s=10, alpha=0.5); ax2.axhline(thr, color='crimson', ls='--')
        ax2.set_xlabel('Granger score'); ax2.set_ylabel('‖ρ‖'); ax2.set_title(f'norm vs prior  (Spearman {rho:+.2f})')
    else:
        ax2.text(0.5, 0.5, 'no graph_score', ha='center'); ax2.axis('off')
    fig.tight_layout(); plt.show()
    print(f"{name}: {int((edge_norm<thr).sum())}/{E} edges below the Otsu cut ({thr:.2f}) — prune candidates")

### §16b · Scalp colored by bimodal norm group

The same Otsu split as §15, but drawn on the head: low-norm (prune-candidate) edges in grey,
high-norm edges in blue — so the two populations are visible *anatomically* (e.g. whether the
weak group is long-range / inter-hemisphere).

In [ ]:
from matplotlib.collections import LineCollection
from pathlib import Path as _P
from data.lemon_dataset import montage_positions
for name, rec in HARVEST.items():
    R = np.asarray(rec['H']['rho_src'], float); S, E = R.shape[0], R.shape[1]
    edge_norm = rm.frob(R, 'isometry').reshape(S, E).mean(0)
    thr = rm.bimodal_threshold(edge_norm); hi = edge_norm > thr
    ei_np = rec['ei'].detach().cpu().numpy()
    try:
        ch = list(rec['loaders']['train'].dataset.channels)
        pj = _P(rec['loaders']['train'].dataset._samples[0][0]).parent / 'montage_positions.json'
        pos2d = scalp_xy(montage_positions(ch, positions_json=str(pj) if pj.exists() else None))
    except Exception as e:
        print(f'{name}: no montage ({e})'); continue
    fig, ax = plt.subplots(figsize=(6, 6))
    r = np.abs(pos2d).max() * 1.15; th = np.linspace(0, 2*np.pi, 200)
    ax.plot(r*np.cos(th), r*np.sin(th), 'k', lw=1)
    ax.plot([-.09*r, 0, .09*r], [r*.99, r*1.12, r*.99], 'k', lw=1)
    for mask, col, lab in [(~hi, '#c7c7c7', f'low  (n={int((~hi).sum())})'),
                           (hi,  '#1f4e9c', f'high (n={int(hi.sum())})')]:
        segs = [[(pos2d[s, 0], pos2d[s, 1]), (pos2d[d, 0], pos2d[d, 1])]
                for s, d in zip(ei_np[0][mask], ei_np[1][mask])]
        ax.add_collection(LineCollection(segs, colors=col, lw=1.8, alpha=0.85, label=lab))
    ax.scatter(pos2d[:, 0], pos2d[:, 1], s=18, c='k', zorder=3)
    ax.set_aspect('equal'); ax.axis('off'); ax.legend(loc='upper right', fontsize=9)
    ax.set_title(f'{name}: edges by norm mode (Otsu {thr:.2f})')
    fig.tight_layout(); plt.show()

## §17 · Transport metrics — what the map DOES, not how big it is

`‖ρ‖` is signal-free and buries rotation under scale. These measure the edge's *action*:

- **delta energy** `‖delta_e‖` — the disagreement the sheaf actually transmits on the edge.
- **transport ratio** `r_e = ‖delta_e‖ / ‖h_dst − h_src‖` — what the map does to the endpoint
  disagreement. `r<1` aligns/transports, `r>1` contrasts, `r≈1` inert. (Under identity maps `r=1`.)
- **transport twist** `‖Q_dstᵀ Q_src − I‖` — scale-free rotation the edge applies (the polar Q).

If `r_e` clusters near 1 the maps are functionally idle regardless of norm; if it spreads, the
align-vs-contrast split is a real axis `‖ρ‖` could never surface.

In [ ]:
metrics = {}
for name, rec in HARVEST.items():
    H = rec['H']
    de = rm.delta_energy(H); r = rm.transport_ratio(H); tw = rm.edge_transport_twist(H)
    metrics[name] = dict(delta=de, r=r, twist=tw)
    thr = rm.bimodal_threshold(r)
    fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
    axs[0].hist(de, bins=40, color='#6ACC65'); axs[0].set_title(f'{name}: delta energy ‖delta_e‖'); axs[0].set_xlabel('‖delta_e‖')
    axs[1].hist(r, bins=40, color='#4878CF'); axs[1].axvline(1.0, color='k', ls=':', label='r=1 (inert)')
    axs[1].axvline(thr, color='crimson', ls='--', label=f'Otsu {thr:.2f}')
    axs[1].set_title(f'transport ratio r_e  (aligns {int((r<1).sum())} / contrasts {int((r>1).sum())})')
    axs[1].set_xlabel('r_e'); axs[1].legend(fontsize=8)
    axs[2].hist(tw, bins=40, color='#B47CC7'); axs[2].set_title('transport twist ‖Q_dstᵀQ_src − I‖'); axs[2].set_xlabel('twist')
    fig.tight_layout(); plt.show()
    print(f"{name}: r_e median={np.median(r):.2f}  aligns(r<1)={int((r<1).sum())}/{len(r)}  "
          f"twist median={np.median(tw):.2f}")

### §17b · Does the action metric relate to geometry and the prior?

Left: `r_e` vs twist — is 'aligns vs contrasts' a different axis from 'rotates a lot vs little'?
Middle: `r_e` vs the Granger prior — does an action metric correlate where the norm (ρ=0.16) did
not? Right: `r_e` vs `‖ρ‖` — confirm the two metrics are genuinely different.

In [ ]:
from scipy.stats import spearmanr
for name, rec in HARVEST.items():
    m = metrics[name]; r, tw = m['r'], m['twist']
    R = np.asarray(rec['H']['rho_src'], float); S, E = R.shape[0], R.shape[1]
    norm = rm.frob(R, 'isometry').reshape(S, E).mean(0)
    ei_np = rec['ei'].detach().cpu().numpy()
    gs = rec['ckpt'].get('graph_score'); gv = np.asarray(gs)[ei_np[0], ei_np[1]] if gs is not None else None
    fig, axs = plt.subplots(1, 3, figsize=(14, 4.0))
    axs[0].scatter(tw, r, s=12, alpha=0.5); axs[0].axhline(1, color='k', ls=':')
    axs[0].set_xlabel('transport twist'); axs[0].set_ylabel('r_e'); axs[0].set_title(f'r_e vs twist  (Spearman {spearmanr(tw, r)[0]:+.2f})')
    if gv is not None:
        axs[1].scatter(gv, r, s=12, alpha=0.5); axs[1].axhline(1, color='k', ls=':')
        axs[1].set_xlabel('Granger score'); axs[1].set_ylabel('r_e'); axs[1].set_title(f'r_e vs prior  (Spearman {spearmanr(gv, r)[0]:+.2f})')
    else:
        axs[1].axis('off'); axs[1].text(0.5, 0.5, 'no graph_score', ha='center')
    axs[2].scatter(norm, r, s=12, alpha=0.5); axs[2].axhline(1, color='k', ls=':')
    axs[2].set_xlabel('‖ρ‖ [isometry]'); axs[2].set_ylabel('r_e'); axs[2].set_title(f'r_e vs norm  (Spearman {spearmanr(norm, r)[0]:+.2f})')
    fig.suptitle(f'{name}: transport ratio vs geometry / prior / norm'); fig.tight_layout(); plt.show()

### §17c · Scalp by transport action

Left: edges colored by raw `r_e` (shared scale). Right: the Otsu split of `r_e` on the head —
the two transport populations anatomically (e.g. whether the strong-transport group is local vs
long-range), the metric-swapped version of §16b.

In [ ]:
from matplotlib.collections import LineCollection
from pathlib import Path as _P
from data.lemon_dataset import montage_positions
for name, rec in HARVEST.items():
    r = metrics[name]['r']; thr = rm.bimodal_threshold(r); hi = r > thr
    ei_np = rec['ei'].detach().cpu().numpy()
    try:
        ds = rec['loaders']['train'].dataset
        pj = _P(ds._samples[0][0]).parent / 'montage_positions.json'
        pos2d = scalp_xy(montage_positions(list(ds.channels), positions_json=str(pj) if pj.exists() else None))
    except Exception as e:
        print(f'{name}: no montage ({e})'); continue
    fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 6))
    lc = draw_topo_edges(a0, ei_np, pos2d, r, f'{name}: transport ratio r_e (raw)', scale01=False, cmap='coolwarm')
    fig.colorbar(lc, ax=a0, fraction=0.046, label='r_e')
    rr = np.abs(pos2d).max() * 1.15; th = np.linspace(0, 2*np.pi, 200)
    a1.plot(rr*np.cos(th), rr*np.sin(th), 'k', lw=1); a1.plot([-.09*rr, 0, .09*rr], [rr*.99, rr*1.12, rr*.99], 'k', lw=1)
    for mask, col, lab in [(~hi, '#c7c7c7', f'low r  (n={int((~hi).sum())})'), (hi, '#b2182b', f'high r (n={int(hi.sum())})')]:
        segs = [[(pos2d[s, 0], pos2d[s, 1]), (pos2d[d, 0], pos2d[d, 1])] for s, d in zip(ei_np[0][mask], ei_np[1][mask])]
        a1.add_collection(LineCollection(segs, colors=col, lw=1.8, alpha=0.85, label=lab))
    a1.scatter(pos2d[:, 0], pos2d[:, 1], s=18, c='k', zorder=3); a1.set_aspect('equal'); a1.axis('off'); a1.legend(loc='upper right', fontsize=9)
    a1.set_title(f'r_e modes (Otsu {thr:.2f})')
    fig.tight_layout(); plt.show()

## §18 · Is the transport real, and are we actually changing anything?

Two checks on the transport-ratio story. **(a)** `r_e` vs a random-map null: random rotations at
the same scale destroy neighbor similarity, so the null sits well above 1 — a learned `r_e` far
below it is genuine tuned transport, not shrinkage. **(b)** `r_e` vs homophily (`cos(h_src,h_dst)`)
and vs electrode distance — is alignment easier for similar / nearby endpoints?

In [ ]:
from scipy.stats import spearmanr
for name, rec in HARVEST.items():
    H = rec['H']; r = rm.transport_ratio(H); rn = rm.transport_ratio_null(H)
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.hist(rn, bins=40, alpha=0.55, color='grey', density=True, label=f'random-map null (med {np.median(rn):.2f})')
    ax.hist(r,  bins=40, alpha=0.75, color='#4878CF', density=True, label=f'learned (med {np.median(r):.2f})')
    ax.axvline(1.0, color='k', ls=':', label='r=1 (inert)')
    ax.set_xlabel('r_e'); ax.set_yticks([]); ax.legend(fontsize=8)
    ax.set_title(f'{name}: transport ratio vs random-map floor')
    fig.tight_layout(); plt.show()
    print(f"{name}: learned median {np.median(r):.2f}  vs null {np.median(rn):.2f}  "
          f"(learned is {np.median(rn)/max(np.median(r),1e-9):.1f}x below the random floor)")

In [ ]:
from scipy.stats import spearmanr
from pathlib import Path as _P
from data.lemon_dataset import montage_positions
for name, rec in HARVEST.items():
    H = rec['H']; r = rm.transport_ratio(H)
    ncos = rm.neighbor_cosine(H['h_src'], H['h_dst']).mean(0)      # (E,) homophily
    ei_np = rec['ei'].detach().cpu().numpy(); pairs = np.stack([ei_np[0], ei_np[1]], 1)
    dist = None
    try:
        ds = rec['loaders']['train'].dataset
        pj = _P(ds._samples[0][0]).parent / 'montage_positions.json'
        pos = montage_positions(list(ds.channels), positions_json=str(pj) if pj.exists() else None)
        dist = rm.edge_distance(pairs, pos)
    except Exception as e:
        print(f'{name}: no positions ({e})')
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.0))
    axs[0].scatter(ncos, r, s=12, alpha=0.5); axs[0].axhline(1, color='k', ls=':')
    axs[0].set_xlabel('neighbor cosine (homophily)'); axs[0].set_ylabel('r_e')
    axs[0].set_title(f'r_e vs homophily  (Spearman {spearmanr(ncos, r)[0]:+.2f})')
    if dist is not None:
        axs[1].scatter(dist, r, s=12, alpha=0.5); axs[1].axhline(1, color='k', ls=':')
        axs[1].set_xlabel('electrode distance'); axs[1].set_ylabel('r_e')
        axs[1].set_title(f'r_e vs distance  (Spearman {spearmanr(dist, r)[0]:+.2f})')
    else:
        axs[1].axis('off')
    fig.suptitle(f'{name}: what makes an edge align vs contrast?'); fig.tight_layout(); plt.show()

### §18b · Are we really changing things? (per-edge message → per-node effect)

`delta` is edge-space; what reaches the state is the pulled-back message `msg_e = delta·ρ_dstᵀ`,
summed into `sheaf_h = h_t − step·Σ msg`. So the honest test is the **per-node relative change**
`‖sheaf_h − h_t‖ / ‖h_t‖`: ~0 means the graph is inert *there* regardless of any map statistic;
sizeable means it genuinely reshapes the encoding. The scalp shows where (if anywhere) it acts.

In [ ]:
from pathlib import Path as _P
from data.lemon_dataset import montage_positions
for name, rec in HARVEST.items():
    H = rec['H']
    me = rm.message_energy(H)                       # (E,) per-edge node-space push
    nsc = rm.node_state_change(H)                   # (N,) per-node relative change
    glob = rm.sheaf_relative_change(H['sheaf_h'], H['h_t']).mean()
    ei_np = rec['ei'].detach().cpu().numpy()
    try:
        ds = rec['loaders']['train'].dataset
        pj = _P(ds._samples[0][0]).parent / 'montage_positions.json'
        pos2d = scalp_xy(montage_positions(list(ds.channels), positions_json=str(pj) if pj.exists() else None))
    except Exception as e:
        pos2d = None; print(f'{name}: no montage ({e})')
    fig = plt.figure(figsize=(12, 4.2))
    a0 = fig.add_subplot(1, 2, 1)
    a0.hist(me, bins=40, color='#6ACC65'); a0.set_xlabel('per-edge message energy ‖msg_e‖'); a0.set_ylabel('edges')
    a0.set_title(f'{name}: message energy  (net ‖sheaf_h−h_t‖/‖h_t‖ = {glob:.3f})')
    if pos2d is not None:
        a1 = fig.add_subplot(1, 2, 2)
        rr = np.abs(pos2d).max() * 1.15; th = np.linspace(0, 2*np.pi, 200)
        a1.plot(rr*np.cos(th), rr*np.sin(th), 'k', lw=1); a1.plot([-.09*rr, 0, .09*rr], [rr*.99, rr*1.12, rr*.99], 'k', lw=1)
        for s, d in zip(ei_np[0], ei_np[1]):
            a1.plot([pos2d[s, 0], pos2d[d, 0]], [pos2d[s, 1], pos2d[d, 1]], color='#cccccc', lw=0.5, zorder=1)
        sc = a1.scatter(pos2d[:, 0], pos2d[:, 1], c=nsc, s=180, cmap='inferno', zorder=3, edgecolors='k', linewidths=0.5)
        a1.set_aspect('equal'); a1.axis('off'); a1.set_title('per-node change ‖sheaf_h−h_t‖/‖h_t‖')
        fig.colorbar(sc, ax=a1, fraction=0.046, label='relative change')
    fig.tight_layout(); plt.show()
    print(f"{name}: net encoding change = {glob:.3f}  (0 = inert graph);  per-node change "
          f"min {nsc.min():.3f} / med {np.median(nsc):.3f} / max {nsc.max():.3f}")